# Pipeline Big Data XAUUSD

Exécuter les cellules dans l’ordre avec un noyau Python sous Linux ou Ubuntu WSL2. Dépendances du noyau : `pyspark==4.0.1 pyarrow psutil matplotlib ipykernel` et Java 17. Le mode par défaut **SNAPSHOT** consulte les manifestes. **TEST** écrit sous `data/pipeline_runs/test`. **PILOT** traite un intervalle UTC borné. **RESUME** reprend les checkpoints du mode configuré. **FULL** cible `data/silver`, `data/gold` et `data/datasets` après les contrôles d’accès aux sources.

Les preuves Bronze du 24 novembre 2026 servent de référence. La couverture conservatrice de 744 689 464 lignes n’est pas un effectif Silver. Les originaux sont lus sur place. Le notebook ne lance ni collecte externe ni entraînement.

## 01 · Configuration et ressources

La date de fin est exclue. Pour reprendre une étape, définir `XAUUSD_STEPS` avec des numéros séparés par des virgules et conserver la même configuration des données. Les manifestes scellent la configuration et les entrées des résultats persistés.

Sur ce poste, ouvrir le projet dans VS Code sous WSL2 et sélectionner le noyau **Python (XAUUSD PySpark WSL2)**. Le sélecteur de mode de la première cellule accepte aussi une valeur saisie directement dans la configuration.

In [ ]:
from collections import defaultdict, deque
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from hashlib import sha256
from importlib.util import find_spec
from pathlib import Path
from time import perf_counter
from uuid import uuid4
import csv, gzip, json, math, os, platform, re, shutil, subprocess, sys
from IPython.display import Markdown, display

def locate_project():
    candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / 'OneDrive/Desktop/Multi-Agents-Trading',
                  *sorted(Path('/mnt/c/Users').glob('*/OneDrive/Desktop/Multi-Agents-Trading'))]
    return next((p for p in candidates if (p / 'data').is_dir() and (p / 'github').is_dir()), Path.cwd())

SMA = EMA = [5, 10, 20, 50]
CFG = dict(
    mode=os.getenv('XAUUSD_MODE', 'SNAPSHOT').upper(),
    resume_mode=os.getenv('XAUUSD_RESUME_MODE', 'PILOT').upper(),
    root=Path(os.getenv('XAUUSD_PROJECT_DIR', str(locate_project()))).resolve(),
    start=os.getenv('XAUUSD_FROM', '2025-08-05'), end=os.getenv('XAUUSD_TO', '2025-08-06'),
    scratch=Path(os.getenv('XAUUSD_SPARK_TMP', str(Path.home() / '.cache/xauusd-spark'))).resolve(),
    steps=set(os.getenv('XAUUSD_STEPS', '02,03,04,05,06,07,08,09,10,11,12,13,14,15,16,17,18,19,20').split(',')),
    source_ids=['historical_old', 'historical_recent', 'touhid', 'carlos', 'jforex'],
    timeframes=['M1','M5','M15','M30','H1','H4','D1','W1','MN1'],
    main='M5', higher=['M15','H1','H4','D1'],
    sma=SMA, ema=EMA, rsi=[7,14], macd=[(5,13,4),(5,26,4)], stoch=[(9,3),(14,3)],
    cci=[14], atr=[7,14], bollinger=[(20,2)], obv=[10,20],
    extra={'price','trend','momentum','volatility','statistics','microstructure','time'},
    extra_window=20, roc=10, adx=14, sar_step=.02, sar_max=.2,
    max_gap_seconds=180, volume_required=False, allowed_quality=['OK'], instrument='XAUUSD',
    dataset_sources=['historical_old','historical_recent','touhid','carlos','jforex','fixture'],
    dataset_start=None, dataset_end=None, features=['sma_5','rsi_7','tick_obv_delta_10'],
    split=None, scaler='standard', lookback=1, benchmark_full=os.getenv('XAUUSD_BENCHMARK_FULL','0')=='1',
    benchmark_levels=os.getenv('XAUUSD_BENCHMARK_LEVELS','day').split(','),
    # Ajouter des périodes HistData uniquement après rapprochement prix/horloge documenté.
    histdata_windows=[('2021-01-05','2021-01-06',5),('2025-08-05','2025-08-06',4)],
    # Ces deux fenêtres HistData servent à la comparaison, leur admission se règle via source_ids.
)
data = CFG['root'] / 'data'
started_at = perf_counter()
mode = CFG['mode']
run_mode = CFG['resume_mode'] if mode in {'RESUME','SNAPSHOT'} else mode
assert run_mode in {'TEST','PILOT','FULL'} and mode in {'TEST','PILOT','FULL','RESUME','SNAPSHOT'}
start, end = map(date.fromisoformat, (CFG['start'], CFG['end']))
if run_mode == 'FULL':
    start, end = date(2004,1,1), date(2026,11,24)
if run_mode == 'TEST':
    start, end = date(2025,8,5), date(2025,8,6)
assert start < end
if CFG['instrument']!='XAUUSD':
    raise ValueError('Le catalogue installé couvre XAUUSD.')
out = data if run_mode == 'FULL' else data / 'pipeline_runs' / (
    'test' if run_mode == 'TEST' else f'pilot_{start}_{end}')
reports = out / 'reports' / 'spark_xauusd'
manifests = out / 'manifests' / 'spark_xauusd'
assert 'onedrive' not in str(CFG['scratch']).lower() and not CFG['scratch'].is_relative_to(data)

def say(text):
    display(Markdown(text))

def active(step):
    return mode != 'SNAPSHOT' and step in CFG['steps']

def probe(command):
    try:
        result = subprocess.run(command, text=True, encoding='utf8', errors='replace', capture_output=True, timeout=7)
        return ((result.stdout or result.stderr).replace('\x00','').strip().splitlines() or [''])[0] if result.returncode == 0 else 'indisponible'
    except (OSError, subprocess.TimeoutExpired):
        return 'indisponible'

psutil = __import__('psutil') if find_spec('psutil') else None
ram = psutil.virtual_memory() if psutil else None
cores = len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else os.cpu_count() or 2
avail = ram.available / 2**30 if ram else 4
workers = min(cores, max(2, int(avail // 1.5)), 6)
driver_gib = max(1, min(6, int(avail * .55)))
shuffle_partitions = max(128,workers*32) if run_mode=='FULL' else max(8,workers*4)
cpu_name = os.environ.get('PROCESSOR_IDENTIFIER') or platform.processor() or platform.machine()
if Path('/proc/cpuinfo').is_file():
    cpu_name = next((line.split(':',1)[1].strip() for line in Path('/proc/cpuinfo').read_text().splitlines()
                     if line.startswith('model name')),cpu_name)
spark_options = {'spark.driver.memory': f'{driver_gib}g', 'spark.sql.shuffle.partitions': str(shuffle_partitions),
                 'spark.sql.adaptive.enabled':'true','spark.sql.files.maxPartitionBytes':str(96*2**20),
                 'spark.local.dir':str(CFG['scratch']), 'spark.sql.session.timeZone':'UTC',
                 'spark.driver.bindAddress':'127.0.0.1','spark.driver.host':'127.0.0.1',
                 'spark.sql.parquet.compression.codec':'zstd',
                 'spark.sql.parquet.outputTimestampType':'TIMESTAMP_MICROS',
                 'spark.sql.execution.arrow.pyspark.enabled':'false'}
free_data = shutil.disk_usage(data if data.exists() else Path.cwd()).free / 2**30
print(dict(mode=mode, period=[str(start),str(end)], CPU=cpu_name,
           cores=cores, ram_total_GiB=round(ram.total/2**30,2) if ram else None,
           ram_available_GiB=round(avail,2), disk_available_GiB=round(free_data,2),
           Python=platform.python_version(), Java=probe(['java','-version']),
           PySpark=bool(find_spec('pyspark')), Docker=probe(['docker','--version']),
           WSL2='microsoft' in platform.release().lower() or probe(['wsl.exe','--list','--verbose'])!='indisponible',
           master=f'local[{workers}]', scratch=str(CFG['scratch'])))
spark = None
if mode != 'SNAPSHOT':
    if not find_spec('pyspark'):
        raise RuntimeError('Installer PySpark et Java 17 dans le noyau Linux/WSL2.')
    if os.name == 'nt':
        raise RuntimeError('Utiliser le noyau Linux/WSL2 pour les écritures Parquet Spark.')
    from pyspark.sql import SparkSession, Window, functions as F, types as T
    from pyspark import StorageLevel
    CFG['scratch'].mkdir(parents=True, exist_ok=True)
    os.environ['TZ']='UTC'
    import time
    if hasattr(time,'tzset'):
        time.tzset()
    os.environ.setdefault('SPARK_LOCAL_IP','127.0.0.1')
    os.environ.setdefault('PYSPARK_PYTHON',sys.executable)
    spark = SparkSession.builder.master(f'local[{workers}]').appName('XAUUSD Medallion').config(
        map=spark_options).getOrCreate()
    spark.sparkContext.setLogLevel('ERROR')
    print(f'Spark {spark.version} · UI {spark.sparkContext.uiWebUrl}')

def read_json(path, default=None):
    return json.loads(path.read_text(encoding='utf-8-sig')) if path.is_file() else default

def put_json(path, value):
    path.parent.mkdir(parents=True,exist_ok=True)
    temp = path.with_name(f'.{path.name}.{uuid4().hex}.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str, allow_nan=False)+'\n', encoding='utf8')
    temp.replace(path)

def hash_file(path):
    # Sur WSL, un gros fichier déjà sur NTFS est haché par Windows sans traverser DrvFS octet par octet.
    if (path.stat().st_size>=2**30 and re.match(r'^/mnt/[a-z]/',str(path))
        and shutil.which('powershell.exe') and shutil.which('wslpath')):
        import base64
        try:
            win = subprocess.check_output(['wslpath','-w',str(path)],text=True,timeout=10).strip()
            command = "$ErrorActionPreference='Stop'; $s=[System.IO.File]::OpenRead('" + win.replace("'","''") + "'); try { $sha=[System.Security.Cryptography.SHA256]::Create(); [Console]::WriteLine([BitConverter]::ToString($sha.ComputeHash($s)).Replace('-','')) } finally { $s.Dispose() }"
            encoded = base64.b64encode(command.encode('utf-16-le')).decode('ascii')
            raw = subprocess.check_output(['powershell.exe','-NoProfile','-NonInteractive',
                                          '-EncodedCommand',encoded],stderr=subprocess.DEVNULL,timeout=900)
            match=re.search(rb'(?i)[0-9a-f]{64}',raw)
            if match:
                return match.group().decode('ascii').lower()
        except (OSError,subprocess.SubprocessError):
            pass
    with path.open('rb') as stream:
        from hashlib import file_digest
        return file_digest(stream,'sha256').hexdigest()

def digest(value):
    return sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()

key_config = {k: sorted(v) if isinstance(v,set) else v for k,v in CFG.items()
              if k not in {'mode','resume_mode','root','scratch','steps','benchmark_full','benchmark_levels','start','end'}}
run_key = digest([key_config,str(start),str(end),'xauusd-spark-v1'])

def manifest(name):
    return read_json(manifests / (name+'.json'), {})

def completed(name):
    state = manifest(name)
    return state.get('run_key') == run_key and state.get('status') == 'complete'

def status(name):
    record = manifest(name)
    say(f"**{name}** · {record.get('status','aucun résultat sauvegardé')}.")
    if record:
        print({k:v for k,v in record.items() if k not in {'run_key','files','schema'}})

def require(name, path):
    if not completed(name) or not path.is_dir():
        raise RuntimeError(f'Checkpoint {name} absent ou incompatible : {path}')
    return spark.read.parquet(str(path))

def publish(df, path, partition=()):
    if path.exists():
        if read_json(path/'_pipeline.json',{}).get('run_key')==run_key and (path/'_SUCCESS').exists():
            return
        raise FileExistsError(f'Résultat existant sans manifeste compatible : {path}')
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.{uuid4().hex}.pending')
    writer = df.write.mode('errorifexists').option('compression','zstd')
    if partition:
        writer = writer.partitionBy(*partition)
    writer.parquet(str(tmp))
    if not (tmp / '_SUCCESS').exists():
        raise RuntimeError(f'Écriture Spark incomplète : {tmp}')
    put_json(tmp/'_pipeline.json',{'run_key':run_key})
    tmp.rename(path)

def finish(name, **stats):
    put_json(manifests / (name+'.json'), {'run_key':run_key,'status':'complete','elapsed_run_seconds':round(perf_counter()-started_at,3),**stats})
    status(name)

NORM = out/'silver/normalized_sources'
QUAR = out/'quarantine/spark_xauusd'
TICKS = out/'silver/ticks'
BARS = out/'silver/bars'
GOLD = out/'gold/features'
BUILT = out/'datasets/spark_xauusd'
say('La configuration et les chemins s’appliquent à toutes les cellules suivantes.')

## 02 · Benchmark PySpark

Le benchmark lit des cotations réelles, les regroupe par seconde et écrit un résultat ZSTD temporaire. Les mesures portent sur le périmètre lu, avec débit, pic mémoire, CPU, disque temporaire, shuffle et fichiers produits. Le mode TEST réserve le benchmark aux tests de traitement. Les paliers au-delà de la journée s’activent avec `XAUUSD_BENCHMARK_LEVELS=day,week,month,year`.

Le code du palier historique complet est en section 20. L’option `XAUUSD_BENCHMARK_FULL=1` s’applique au mode FULL lorsque la validation Silver a abouti.

In [ ]:
from tempfile import TemporaryDirectory
from threading import Event, Thread
from urllib.parse import quote
from urllib.request import urlopen

def size_of(path):
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file())

def stages():
    try:
        app = quote(spark.sparkContext.applicationId)
        with urlopen(f'{spark.sparkContext.uiWebUrl}/api/v1/applications/{app}/stages',timeout=3) as stream:
            return {row['stageId']:row for row in json.load(stream)}
    except (OSError, ValueError):
        return None

def benchmark(paths, begin, finish_at, partitions, frame=None):
    original = spark.conf.get('spark.sql.shuffle.partitions')
    spark.conf.set('spark.sql.shuffle.partitions', str(partitions))
    before = stages()
    stop = Event()
    peak = dict(ram_bytes=0, scratch_bytes=0)
    cpu0 = psutil.cpu_times() if psutil else None
    def monitor():
        while not stop.is_set():
            if psutil:
                with contextmanager_suppressed(psutil.Error):
                    parent = psutil.Process()
                    peak['ram_bytes'] = max(peak['ram_bytes'], sum(
                        p.memory_info().rss for p in [parent,*parent.children(recursive=True)]))
            with contextmanager_suppressed(OSError):
                peak['scratch_bytes'] = max(peak['scratch_bytes'], size_of(CFG['scratch']))
            stop.wait(.25)
    timer = perf_counter()
    try:
        with TemporaryDirectory(prefix='benchmark_',dir=CFG['scratch']) as folder:
            tmp = Path(folder)
            thread = Thread(target=monitor,daemon=True)
            thread.start()
            try:
                if frame is None:
                    rows = spark.read.option('header',True).schema(
                        'timestamp string, ask string, bid string, ask_volume string, bid_volume string').csv(paths)
                else:
                    rows = frame.select(F.date_format('timestamp_utc','yyyy-MM-dd HH:mm:ss.SSS').alias('timestamp'))
                rows = rows.where((F.col('timestamp')>=str(begin)) & (F.col('timestamp')<str(finish_at)))
                agg = rows.groupBy(F.substring('timestamp',1,19).alias('second')).count()
                agg.write.option('compression','zstd').parquet(str(tmp/'measure'))
                seconds = perf_counter()-timer
                result = spark.read.parquet(str(tmp/'measure'))
                count = result.agg(F.sum('count')).first()[0] or 0
                parquet_bytes = size_of(tmp/'measure')
                output_parts = len(list((tmp/'measure').glob('*.parquet')))
            finally:
                stop.set()
                thread.join()
    finally:
        spark.conf.set('spark.sql.shuffle.partitions', original)
    after = stages()
    cpu_pct = None
    if cpu0:
        cpu1 = psutil.cpu_times()
        duration = sum(cpu1)-sum(cpu0)
        cpu_pct = round(100*(1-(cpu1.idle-cpu0.idle)/duration),1) if duration>0 else None
    return dict(start=str(begin),end=str(finish_at),ticks=count,seconds=round(seconds,3),
                ticks_per_second=round(count/seconds), cpu_system_pct=cpu_pct, **peak,
                shuffle_bytes=sum(row.get('shuffleWriteBytes',0) for k,row in after.items() if k not in before)
                if before is not None and after is not None else None,
                shuffle_partitions=partitions, parquet_bytes=parquet_bytes, parquet_files=output_parts)

from contextlib import suppress as contextmanager_suppressed
bench_path = reports/'benchmark.json'
if active('02') and run_mode != 'TEST':
    bench = []
    target_years = [2024,2025,2026]
    bench_start = date(2025,8,5) if run_mode=='FULL' else start
    bench_end = date(2025,8,6) if run_mode=='FULL' else end
    levels = CFG['benchmark_levels']
    for level in levels:
        dates = {'day':(bench_start,min(bench_end,bench_start+timedelta(days=1))),
                 'week':(bench_start,bench_start+timedelta(days=7)),
                 'month':(bench_start.replace(day=1),(bench_start.replace(day=28)+timedelta(days=4)).replace(day=1)),
                 'year':(date(bench_start.year,1,1),date(bench_start.year+1,1,1))}
        a,b = dates[level]
        # L’édition Touhid couvre cette fenêtre, distincte du périmètre historique complet.
        b = min(b,date(2026,11,23))
        paths = []
        for y in target_years:
            folder = data/'recovered/autres_fournisseurs/touhid_xauusd'/str(y)
            for rec in (read_json(folder/'audit_annuel.json',{}) or {}).get('files',[]):
                if rec['first_utc'][:10] < str(b) and rec['last_utc'][:10] >= str(a):
                    paths.append(str(folder/'brut'/rec['file']))
        if not paths:
            print(f'{level} : aucun fichier Touhid dans la fenêtre {a}–{b}')
            continue
        for count in ([max(4,workers),int(spark_options['spark.sql.shuffle.partitions'])] if level=='day'
                      else [int(spark_options['spark.sql.shuffle.partitions'])]):
            row = benchmark(paths,a,b,count) | {'level':level}
            if level=='year' and a.year in (2024,2025):
                expected=read_json(data/'recovered/autres_fournisseurs/touhid_xauusd'/str(a.year)/'audit_annuel.json')['ticks']
                assert row['ticks']==expected
            bench.append(row)
            print(row)
        if level=='day':
            assert len({row['ticks'] for row in bench if row['level']=='day'})==1
    if bench:
        put_json(bench_path,bench)
else:
    bench = read_json(bench_path,[])
    print(f'Mesures sauvegardées : {len(bench)}')
say('Les paliers mesurent la lecture et l’agrégation du flux Touhid. La section 20 peut mesurer l’historique Silver complet après validation.')

## 03 · Bronze et catalogue

Le catalogue reprend les SHA-256 déjà consignés et distingue éditeur, fournisseur déclaré, unités et statut. Les lignes d’une même période peuvent apparaître dans plusieurs sources.

In [ ]:
D = data/'recovered/autres_fournisseurs'
legacy = {'historical_old':'XAUUSD_Ticks_2004.01.01_2021.10.15.csv',
          'historical_recent':'XAUUSD_Ticks_2021.01.04_2026.11.20.csv'}
legacy_zips = read_json(data/'manifests/archives_historiques.json',[])
legacy_refs = read_json(data/'source_runs/audit_local/origines_sha256.json',[])
import base64, zlib
profile_notebook=read_json(CFG['root']/'github/Profilage_donnees_XAUUSD.ipynb',{})
old_payload=profile_notebook.get('metadata',{}).get('xauusd',{}).get('audit_base64_zlib')
original_audit=json.loads(zlib.decompress(base64.b64decode(''.join(old_payload)))) if old_payload else {}
annual = []
annual_path = data/'reports/Couverture_annuelle_XAUUSD.csv'
if annual_path.exists():
    with annual_path.open(encoding='utf-8-sig',newline='') as stream:
        annual=list(csv.DictReader(stream,delimiter=';'))

def catalog_item(source,path,fmt,count,first,last,clock,hash_value,status,distributor,provider,license_text,volume):
    return dict(source_id=source,provider=provider,distributor=distributor,source_file_id=path.relative_to(data).as_posix(),
                format=fmt,ticks=count,first=first,last=last,clock=clock,price_unit='USD par once troy',
                volume_unit=volume,sha256=hash_value,validation=status,license=license_text)

def catalogue():
    result = []
    for item in legacy_refs:
        source = next((s for s,n in legacy.items() if n==item['file']),None)
        if not source:
            continue
        result.append(catalog_item(source,data/item['file'],'csv_decimal_comma',
                      original_audit.get('ancien' if source=='historical_old' else 'recent',{}).get('controle',{}).get('lignes_terminees'),
                      '2004-01-01' if source=='historical_old' else '2021-01-04',
                      '2021-10-15' if source=='historical_old' else '2026-11-20','EET/EEST',item['sha256'],
                      'audit historique','Export local','Dukascopy (origine documentée)','Vérifier les droits de redistribution',
                      'Volumes de cotation, échelle 1 testée'))
    for path in sorted(D.glob('touhid_xauusd/*/audit_annuel.json')):
        for rec in read_json(path)['files']:
            result.append(catalog_item('touhid',path.parent/'brut'/rec['file'],'csv.gz',rec['ticks'],
                          rec['first_utc'],rec['last_utc'],'UTC',rec['sha256'],
                          'audité, rapprochement échantillonné','Touhid-HZ','Dukascopy déclaré',
                          'MIT pour le conditionnement, usage recherche selon éditeur','Millions, cotation'))
    for path in sorted(D.glob('carlos_silva_xauusd/year=*/audit_annuel.json')):
        audit = read_json(path)
        for rec in audit['files']:
            relative = Path(rec['file'])
            month = relative.parts[1].split('=')[1]
            y = int(path.parent.name.split('=')[1])
            first = date(y,int(month),1)
            last = (first.replace(day=28)+timedelta(days=4)).replace(day=1)-timedelta(days=1)
            result.append(catalog_item('carlos',path.parent.parent/relative,'parquet',None,str(first),str(last),
                          'UTC',rec['sha256'],'audité par année, rapprochement échantillonné','Carlos Silva',
                          'Flux de courtage, fournisseur non confirmé','CC BY 4.0 déclarée','Volumes de cotation, échelle testée'))
    for path in sorted(D.glob('histdata/*/*/HISTDATA_*.audit.json')):
        audit = read_json(path)
        rec = next(row for row in audit['files'] if row['file'].endswith('.csv'))
        first = datetime.strptime(audit['first_raw'][:8],'%Y%m%d').date()
        last = datetime.strptime(audit['last_raw'][:8],'%Y%m%d').date()
        result.append(catalog_item('histdata',path.parent/'brut'/rec['file'],'histdata',audit['ticks'],str(first),
                      str(last),'Horloge brute',rec['sha256'],'audité, UTC partiel','HistData','HistData',
                      'Archives publiques, droits à vérifier','Indisponibles'))
    for path in sorted((data/'recovered/dukascopy').glob('*/*/*_JForex_UTC.csv')):
        day = re.search(r'\d{4}-\d{2}-\d{2}',path.name).group()
        audit = read_json(data/'manifests'/f'JForex_{day}.audit.json',{})
        result.append(catalog_item('jforex',path,'csv_utc',audit.get('summary',{}).get('ticks'),day,day,'UTC',
                      audit.get('sha256'),audit.get('validation_status','à contrôler'),'JForex SDK','Dukascopy',
                      'Export officiel, droits à vérifier','Millions, cotation'))
    return result

sources = catalogue()
source_identity = [(r['source_file_id'],r['sha256'], (data/r['source_file_id']).stat().st_size,
                    (data/r['source_file_id']).stat().st_mtime_ns)
                   for r in sources if r['source_id'] in CFG['source_ids'] and (data/r['source_file_id']).is_file()]
run_key = digest([run_key,source_identity,hash_file(data/'reports/preuves_decimales_2021-01-05.json')
                  if (data/'reports/preuves_decimales_2021-01-05.json').exists() else None])
bronze_counts = defaultdict(int)
for item in sources:
    bronze_counts[item['source_id']]+=1
print('Fichiers inventoriés',dict(bronze_counts))
print('Référence Bronze historique déchevauchée',sum(int(row['ticks_locaux']) for row in annual) if annual else None)
if active('03'):
    put_json(reports/'catalogue_bronze.json',{'sources':sources,'archives_originales':legacy_zips})
say('Les chemins sont relatifs à data. Une borne mensuelle dans le catalogue désigne la partition du fournisseur. Les dates exactes se mesurent au profilage.')

## 04 · Profilage Spark et lecture brute

Les lecteurs préservent l’ordre physique au moyen de l’offset d’octet dans les CSV historiques et du rang physique dans les fichiers complémentaires. Le profil agrège par année, mois, jour, heure et minute. Les rejets sont comptés dans les étapes de normalisation et de validation.

In [ ]:
from itertools import product

RAW_SCHEMA = 'source_id string, source_file_id string, source_offset long, raw string, clock string, ask_text string, bid_text string, askv_text string, bidv_text string'

def byte_lower(path,target):
    # Recherche d’une frontière de ligne dans l’export trié. La fenêtre réelle est élargie d’un jour.
    with path.open('rb') as stream:
        stream.readline()
        lo, hi = stream.tell(), path.stat().st_size
        while hi-lo>1024*1024:
            middle=(lo+hi)//2
            stream.seek(middle-1)
            stream.readline()
            pos=stream.tell()
            if pos>=hi:
                hi=middle
                continue
            raw=stream.readline()
            if raw[:23]<target:
                lo=stream.tell()
            else:
                hi=middle
        stream.seek(lo)
        while True:
            pos=stream.tell()
            raw=stream.readline()
            if not raw or raw[:23]>=target:
                return pos

def historical_rows(spark, path, source, relative, a, b):
    left = byte_lower(path,(a-timedelta(days=1)).strftime('%Y.%m.%d').encode())
    right = byte_lower(path,(b+timedelta(days=1)).strftime('%Y.%m.%d').encode())
    # Plages non chevauchantes, avec lecture jusqu’à la fin de la dernière ligne du bloc.
    block = 64*2**20
    ranges = [(offset,min(offset+block,right)) for offset in range(left,right,block)]
    if not ranges:
        return spark.createDataFrame([],RAW_SCHEMA)
    name = str(path)
    def lines(bounds):
        with open(name,'rb') as stream:
            for low,high in bounds:
                stream.seek(low-1)
                if stream.read(1)!=b'\n':
                    stream.readline()
                while stream.tell()<high:
                    offset=stream.tell()
                    raw=stream.readline()
                    if not raw:
                        break
                    yield source,relative,offset,raw.decode('utf8','replace'),None,None,None,None,None
    return spark.createDataFrame(spark.sparkContext.parallelize(ranges,min(len(ranges),max(workers*4,1))).mapPartitions(lines),RAW_SCHEMA)

def companion_rows(spark, listed):
    def lines(items):
        for record in items:
            path,source,relative,fmt=record
            if fmt=='parquet':
                import pyarrow.parquet as pq
                names=['timestamp','ask_price','bid_price','ask_volume','bid_volume']
                index=0
                for batch in pq.ParquetFile(path).iter_batches(batch_size=16384,columns=names):
                    cols=[column.to_pylist() for column in batch.columns]
                    for row in zip(*cols):
                        yield source,relative,index,None,*(str(x) if x is not None else None for x in row)
                        index+=1
            else:
                opener=gzip.open if fmt=='csv.gz' else open
                with opener(path,'rt',encoding='utf-8-sig',newline='') as stream:
                    if fmt!='histdata':
                        next(stream,None)
                    for index,text in enumerate(stream):
                        line=text.rstrip('\r\n')
                        row=line.split(',')
                        if fmt=='histdata' and len(row)==4:
                            stamp,bid,ask,_=row
                            fields=(stamp,ask,bid,None,None)
                        elif fmt!='histdata' and len(row)==5:
                            fields=tuple(row)
                        else:
                            fields=(row[0] if row else None,None,None,None,None)
                        yield source,relative,index,line,*fields
    partitions = max(1,min(len(listed),workers*8))
    return spark.createDataFrame(spark.sparkContext.parallelize(listed,partitions).mapPartitions(lines),RAW_SCHEMA)

def test_raw():
    t = datetime(2025,8,5,tzinfo=timezone.utc)
    rows = []
    for index in range(360):
        key = (t+timedelta(minutes=index,seconds=index%29)).isoformat()
        a = f'{3300+(index%13)/10:.3f}'
        b = f'{3299.8+(index%13)/10:.3f}'
        rows.append(('fixture','fixture_ticks.csv',index,None,key,a,b,'0.001','0.002'))
        if index % 61 == 0:
            rows.append(('fixture','fixture_ticks.csv',1000+index,None,key,a,b,'0.001','0.002'))
    return spark.createDataFrame(rows, RAW_SCHEMA)

def read_raw(a,b):
    if run_mode=='TEST':
        return test_raw()
    records = [r for r in sources if r['source_id'] in CFG['source_ids']
               and (r['source_id']!='histdata' or any(str(a)<stop and str(b)>begin for begin,stop,_ in CFG['histdata_windows']))
               and (r['source_id']!='carlos' or '/year=2023/' in r['source_file_id'])
               and r['first'][:10] < str(b+timedelta(days=1))
               and r['last'][:10] >= str(a-timedelta(days=1))]
    # Les éditions 2021–2022 Carlos sont des références comparatives, la base historique y reste prioritaire.
    if not records:
        raise ValueError(f'Aucun lot éligible entre {a} et {b}')
    for rec in records:
        path = data/rec['source_file_id']
        if not path.is_file():
            raise FileNotFoundError(path)
    others = [(str(data/r['source_file_id']),r['source_id'],r['source_file_id'],r['format'])
              for r in records if r['source_id'] not in legacy]
    frames = [historical_rows(spark,data/r['source_file_id'],r['source_id'],r['source_file_id'],a,b)
              for r in records if r['source_id'] in legacy]
    if others:
        frames.append(companion_rows(spark,others))
    from functools import reduce
    return reduce(lambda left,right:left.unionByName(right),frames)

def span_profile(df,col='timestamp_utc',dimensions=()):
    by = ['source_id',*dimensions]
    daily = df.where(F.col(col).isNotNull()).withColumn('year',F.year(col)).withColumn('month',F.month(col))
    monthly = daily.groupBy(*by,'year','month').agg(F.count('*').alias('ticks'),
        F.countDistinct(F.to_date(col)).alias('days'),
        F.countDistinct(F.date_trunc('hour',col)).alias('hours'),
        F.countDistinct(F.date_trunc('minute',col)).alias('minutes'),
        F.min(col).alias('first_utc'),F.max(col).alias('last_utc'))
    return monthly

def ordered_checks(df):
    samples=df.withColumn('block',F.floor(F.col('source_offset')/100000))
    keys=['source_id','source_file_id','block']
    window=Window.partitionBy(*keys).orderBy('source_offset')
    local=(samples.withColumn('prior',F.lag('timestamp_utc').over(window))
           .withColumn('delta',F.unix_millis('timestamp_utc')-F.unix_millis('prior')))
    blocks=local.groupBy(*keys).agg(F.count_if(F.col('delta')<0).alias('reversals'),
        F.count_if(F.col('delta')>CFG['max_gap_seconds']*1000).alias('gaps'),
        F.max('delta').alias('max_gap_ms'),
        F.min_by('timestamp_utc','source_offset').alias('first'),
        F.max_by('timestamp_utc','source_offset').alias('last'))
    edges=Window.partitionBy('source_id','source_file_id').orderBy('block')
    combined=blocks.withColumn('boundary',F.unix_millis('first')-F.unix_millis(F.lag('last').over(edges)))
    return combined.groupBy('source_id').agg(
        (F.sum('reversals')+F.count_if(F.col('boundary')<0)).alias('reversals'),
        (F.sum('gaps')+F.count_if(F.col('boundary')>CFG['max_gap_seconds']*1000)).alias('gaps'),
        F.max(F.greatest('max_gap_ms','boundary')).alias('max_gap_ms'))

print('Lecteurs disponibles : texte historique, CSV/GZIP, Parquet et pilote synthétique.')
say('Les ruptures sont contrôlées à l’intérieur des blocs et à leurs jonctions. Une coupure peut correspondre à une fermeture de marché.')

## 05 · Nettoyage et reconstructions

Le texte source identifie les lignes à 6–9 composants. Une reconstruction n’est retenue que si sa preuve associe le SHA-256 de la ligne brute, l’horodatage UTC et une solution numérique unique. Les valeurs extrêmes restent présentes et sont signalées par les distributions.

In [ ]:
PRICE='decimal(24,12)'
fields=['ask','bid','ask_volume','bid_volume']
recovery_path=data/'reports/preuves_decimales_2021-01-05.json'
evidence=read_json(recovery_path,{})
proofs = [(r['source_raw_sha256'],r['timestamp_utc'],r['reference_values'])
          for r in evidence.get('repair_evidence',[]) if r['status']=='exact_unique' and r['exact_splits']==1]

def clean(raw, proof_rows=None):
    hist=F.col('source_id').isin(*legacy)
    raw=raw.withColumn('line_text',F.regexp_replace('raw',r'[\r\n]+$',''))
    raw=raw.withColumn('parts',F.split('line_text',',',-1))
    raw=raw.withColumn('components',F.when(hist,F.size('parts')))
    raw=raw.withColumn('clock',F.when(hist,F.try_element_at('parts',F.lit(1))).otherwise(F.col('clock')))
    # Éviter l’index hors bornes en mode ANSI.
    def safe_candidates(parts):
        padded=F.concat(parts,F.array_repeat(F.lit(None).cast('string'),9))
        variants=[]
        for widths in product((1,2),repeat=4):
            pos=1
            values=[]
            for width in widths:
                a=F.element_at(padded,pos+1)
                b=F.element_at(padded,pos+2) if width==2 else None
                ok=a.rlike(r'^[+-]?[0-9]+$')
                text=F.concat(a,F.lit('.'),b) if width==2 else a
                if width==2:
                    ok=ok & b.rlike(r'^[0-9]+$')
                values.append(F.when(ok,text.try_cast(PRICE)))
                pos+=width
            keep=(F.size(parts)==pos)
            for value in values:
                keep=keep & value.isNotNull()
            variants.append(F.when(keep,F.array(*values)))
        return F.filter(F.array(*variants),lambda x:x.isNotNull())
    standard=F.col('line_text').rlike(r'^\d{4}\.\d{2}\.\d{2} .*?,[+-]?\d+,\d+,[+-]?\d+,\d+,[+-]?\d+,\d+,[+-]?\d+,\d+$')
    first=F.when(hist & (F.col('components')==9) & standard,F.array(*[
        F.concat(F.element_at('parts',2+2*i),F.lit('.'),F.element_at('parts',3+2*i)).try_cast(PRICE)
        for i in range(4)]))
    raw=raw.withColumn('standard_values',first)
    # Calcul des possibilités limité aux largeurs atypiques et aux lignes visées par une preuve.
    raw=raw.withColumn('raw_sha',F.when(hist & F.col('components').between(6,8),F.sha2('raw',256)))
    refs=spark.createDataFrame(proofs if proof_rows is None else proof_rows,'raw_sha string, proof_time string, reference array<string>')
    raw=raw.join(F.broadcast(refs),'raw_sha','left')
    raw=raw.withColumn('repair_values',F.when(F.col('reference').isNotNull(),F.transform('reference',lambda x:x.try_cast(PRICE))))
    feasible=F.when(hist & F.col('components').between(6,8),safe_candidates(F.col('parts')))
    raw=raw.withColumn('candidate_count',F.coalesce(F.size(feasible),F.lit(0)))
    raw=raw.withColumn('proof_numeric',F.when(F.col('reference').isNotNull(),
                        F.array_contains(feasible,F.col('repair_values'))).otherwise(F.lit(False)))
    return raw.drop('parts','reference','line_text')

print('Reconstructions documentées disponibles',len(proofs))
say('Les combinaisons décimales possibles sont dénombrées. L’admission d’une ligne ambiguë exige la preuve correspondante.')

## 06 · Normalisation UTC et partitions de travail

L’horloge historique est testée par aller-retour avec les règles Europe/Helsinki, sur les deux décalages observés. Les heures ambiguës ou inexistantes sont isolées. HistData est exclu de la fusion par défaut et son traitement est borné aux journées calibrées en cas d’activation.

In [ ]:
def normalize(cleaned,a,b):
    historical=F.col('source_id').isin(*legacy)
    histdata=F.col('source_id')=='histdata'
    wall=F.try_to_timestamp('clock',F.lit('yyyy.MM.dd HH:mm:ss.SSS'))
    utc2=wall-F.expr('INTERVAL 2 HOURS')
    utc3=wall-F.expr('INTERVAL 3 HOURS')
    legal2=F.from_utc_timestamp(utc2,'Europe/Helsinki')==wall
    legal3=F.from_utc_timestamp(utc3,'Europe/Helsinki')==wall
    utc_historical=F.when(legal2 & ~legal3,utc2).when(legal3 & ~legal2,utc3)
    base=F.try_to_timestamp('clock',F.lit('yyyyMMdd HHmmssSSS'))
    hist_ts=F.lit(None).cast('timestamp')
    for lo,hi,h in CFG['histdata_windows']:
        shifted=base+F.expr(f'INTERVAL {h} HOURS')
        hist_ts=F.when((shifted>=F.lit(lo).cast('timestamp')) & (shifted<F.lit(hi).cast('timestamp')),shifted).otherwise(hist_ts)
    zoned=F.when(historical,utc_historical).when(histdata,hist_ts).otherwise(F.col('clock').try_cast('timestamp'))
    df=cleaned.withColumn('timestamp_utc',zoned)
    proof_ok=F.coalesce(F.col('proof_numeric') & (F.col('proof_time').try_cast('timestamp')==F.col('timestamp_utc')),F.lit(False))
    rebuilt=historical & proof_ok & F.col('standard_values').isNull()
    df=df.withColumn('is_reconstructed',rebuilt)
    values=F.when(rebuilt,F.col('repair_values')).otherwise(F.col('standard_values'))
    names=['ask_text','bid_text','askv_text','bidv_text']
    for i,(name,text) in enumerate(zip(fields,names)):
        df=df.withColumn(name,F.when(historical,F.try_element_at(values,F.lit(i+1))).otherwise(F.col(text).try_cast(PRICE)))
    overlap=original_audit.get('comparaison',{})
    same_old_range=F.lit(False)
    if overlap.get('identiques') and overlap.get('differentes')==0:
        same_old_range=(F.col('source_id')=='historical_old') & F.col('clock').between(overlap['premier'],overlap['dernier'])
    bad_volume=(F.col('ask_volume')<0) | (F.col('bid_volume')<0)
    malformed_num = ~historical & ((F.col('askv_text').isNotNull() & F.col('ask_volume').isNull()) |
                                   (F.col('bidv_text').isNotNull() & F.col('bid_volume').isNull()))
    reason=(F.when(same_old_range,F.lit('confirmed_historical_overlap'))
        .when(F.col('timestamp_utc').isNull(),F.when(historical & wall.isNotNull(),F.lit('ambiguous_or_nonexistent_local_clock'))
              .when(histdata,F.lit('unconfirmed_histdata_clock')).otherwise(F.lit('invalid_timestamp')))
        .when(F.unix_micros('timestamp_utc')%1000!=0,F.lit('submillisecond_clock'))
        .when(historical & values.isNull(),F.lit('unresolved_decimal'))
        .when(F.col('ask').isNull() | F.col('bid').isNull() | malformed_num,F.lit('invalid_numeric'))
        .when((F.col('ask')<=0)|(F.col('bid')<=0),F.lit('nonpositive_price'))
        .when(F.col('ask')<F.col('bid'),F.lit('negative_spread'))
        .when(bad_volume,F.lit('negative_volume')))
    # Les lignes non datables restent visibles, même si la période exacte est inconnue.
    legacy_date=F.regexp_replace(F.substring('clock',1,10),r'\.', '-').try_cast('date')
    fallback=F.when(historical,F.coalesce(F.to_date(wall),legacy_date)).when(histdata,F.to_date(base)).otherwise(
        F.substring('clock',1,10).try_cast('date'))
    # Les lignes historiques illisibles sont affectées à une plage d’octets propre au lot.
    if run_mode!='TEST':
        for source,name in legacy.items():
            path=data/name
            if not path.exists():
                continue
            lo=byte_lower(path,a.strftime('%Y.%m.%d').encode())
            hi=byte_lower(path,b.strftime('%Y.%m.%d').encode())
            fallback=F.when(fallback.isNull() & (F.col('source_id')==source) &
                            (F.col('source_offset')>=lo) & (F.col('source_offset')<hi),
                            F.lit(a)).otherwise(fallback)
    # Une heure locale ambiguë est attribuée à sa date brute. Un horodatage illisible
    # est porté au premier lot sélectionné du fichier, afin d’éviter le double comptage.
    owners={item['source_file_id']: max(start,date.fromisoformat(item['first'][:10]))
            for item in sources}
    if owners:
        first_dates=F.create_map(*[F.lit(x) for name,day in owners.items() for x in (name,str(day))])
        fallback=F.coalesce(fallback,F.element_at(first_dates,'source_file_id').cast('date'))
    in_period=((F.col('timestamp_utc')>=F.lit(str(a)).cast('timestamp')) &
               (F.col('timestamp_utc')<F.lit(str(b)).cast('timestamp'))) | (
               F.col('timestamp_utc').isNull() & (fallback>=F.lit(a)) & (fallback<F.lit(b)))
    # Les jours HistData hors calibration ne sont pas lus comme des ticks UTC admissibles.
    allowed_histdata=(~histdata) | hist_ts.isNotNull() | base.isNull()
    return (df.where(in_period & allowed_histdata).withColumn('reason',reason).withColumn('symbol',F.lit(CFG['instrument']))
         .withColumn('source_row_id',F.concat_ws('#','source_file_id',F.col('source_offset')))
         .withColumn('year',F.year('timestamp_utc')).withColumn('month',F.month('timestamp_utc'))
         .withColumn('day',F.dayofmonth('timestamp_utc'))
         .select('timestamp_utc','symbol',*fields,'source_id','source_file_id','source_row_id','source_offset',
                 'is_reconstructed','components','candidate_count','reason','year','month','day','raw','clock'))

def historical_overlap_proof():
    refs={row['file']:row['sha256'] for row in legacy_refs}
    compare=original_audit.get('comparaison',{})
    assert len(refs)==2 and compare.get('identiques')==43478752 and compare.get('differentes')==0
    assert sum(int(row['ticks_locaux']) for row in annual)==505295814
    for name in legacy.values():
        if hash_file(data/name)!=refs[name]:
            raise ValueError(f'Le chevauchement documenté ne s’applique plus à {name}')

def month_windows(a,b):
    while a<b:
        following=(a.replace(day=28)+timedelta(days=4)).replace(day=1)
        stop=min(b,following)
        yield a,stop
        a=stop

def calendar_evidence():
    from zoneinfo import ZoneInfo
    anchors={}
    archive=read_json(data/'reports/comparaisons_archive_locale_2023_2026.json',{})
    jforex=read_json(data/'reports/comparaisons_verification.json',{})
    early=read_json(data/'source_runs/jforex_pilote_2004-01-05/2004-01-05.audit.json',{})
    for day,row in archive.items():
        matches=row.get('best',[])
        if len(matches)==1:
            anchors[day]=matches[0]['offset_minutes_local_minus_utc']
    for day,row in jforex.items():
        if row.get('best_comparison'):
            anchors[day]=row['best_comparison']['offset_minutes_local_minus_utc']
    if len(early.get('best_matches',[]))==1:
        anchors['2004-01-05']=early['best_matches'][0]['offset_minutes_local_minus_utc']
    tz=ZoneInfo('Europe/Helsinki')
    result=[]
    for day,observed in sorted(anchors.items()):
        instant=datetime.fromisoformat(day).replace(hour=12,tzinfo=timezone.utc)
        expected=int(instant.astimezone(tz).utcoffset().total_seconds()/60)
        result.append(dict(day=day,observed_minutes=observed,iana_minutes=expected,match=observed==expected))
    return result

if active('06'):
    clock_checks=calendar_evidence()
    if any(not row['match'] for row in clock_checks):
        raise ValueError(f'Une sonde locale contredit le fuseau paramétré : {clock_checks}')
    put_json(reports/'preuves_fuseau.json',clock_checks)
    print(f'Journées historiques comparées au fuseau : {len(clock_checks)}')
    expected_legacy = run_mode!='TEST' and start<date(2021,10,16) and end>date(2021,1,3)
    if expected_legacy and not completed('legacy_identity'):
        historical_overlap_proof()
        finish('legacy_identity',identical_overlap=43478752)
    for a,b in month_windows(start,end):
        name=f'normalized_{a}'
        path=NORM/f'chunk={a}'
        if not completed(name):
            profile = normalize(clean(read_raw(a,b)),a,b)
            publish(profile,path)
            finish(name,path=str(path.relative_to(out)))
    finish('normalized',chunks=[str(a) for a,_ in month_windows(start,end)])
else:
    status('normalized')
say('Les lignes rejetées restent dans le checkpoint avec leur texte brut, leur source et leur motif.')

## 07 · Réconciliation et déduplication

La clé comprend l’horodatage, les deux prix et les deux volumes. Pour une même clé, la multiplicité retenue est le maximum observé dans un canal. La priorité ne départage que les copies de la même occurrence. Les autres cotations sont conservées et les conflits entre canaux sont signalés.

In [ ]:
priority={'historical_old':0,'historical_recent':0,'jforex':1,'touhid':2,'carlos':3,'histdata':4,'fixture':0}
IDENTITY=['symbol','timestamp_utc',*fields]

def reconcile(accepted):
    priorities=F.create_map(*[F.lit(x) for source,rank in priority.items() for x in (source,rank)])
    values=accepted.withColumn('priority',F.element_at(priorities,'source_id'))
    physical=Window.partitionBy(*IDENTITY,'source_id').orderBy('source_file_id','source_offset')
    numbered=values.withColumn('occurrence',F.row_number().over(physical))
    occurrence=Window.partitionBy(*IDENTITY,'occurrence')
    ranked=occurrence.orderBy('priority','source_id','source_file_id','source_offset')
    numbered=(numbered.withColumn('matching_sources',F.sort_array(F.collect_set('source_id').over(occurrence)))
              .withColumn('pick',F.row_number().over(ranked)))
    selected=numbered.where('pick=1').drop('pick')
    if 'histdata' in CFG['source_ids']:
        price_key=['symbol','timestamp_utc','ask','bid']
        price_copies=Window.partitionBy(*price_key)
        hist_order=Window.partitionBy(*price_key,'source_id').orderBy('source_file_id','source_offset')
        selected=(selected.withColumn('has_quote',F.count_if(F.col('source_id')!='histdata').over(price_copies))
                  .withColumn('price_rank',F.row_number().over(hist_order))
                  .where((F.col('source_id')!='histdata') | (F.col('price_rank')>F.col('has_quote')))
                  .drop('has_quote','price_rank'))
    # Une vue dédupliquée par identité et occurrence conserve l’ordre de chaque source connue.
    order=Window.partitionBy('symbol','timestamp_utc').orderBy('priority','source_id','source_file_id','source_offset',*fields)
    times=Window.partitionBy('symbol','timestamp_utc')
    variants=selected.withColumn('source_variants',F.size(F.array_distinct(F.flatten(
        F.collect_list('matching_sources').over(times)))))
    variants=variants.withColumn('quote_variant',F.dense_rank().over(times.orderBy(*fields)))
    multi=F.max('quote_variant').over(times)>1
    conflict=(F.col('source_variants')>1) & multi
    return (variants.withColumn('tick_sequence',F.row_number().over(order))
            .withColumn('quality_flag',F.when(conflict,'CONFLICT').when(F.col('source_id')=='histdata','PRICE_ONLY').otherwise('OK'))
            .withColumn('mid_price',((F.col('ask')+F.col('bid'))/2).cast('decimal(26,13)'))
            .withColumn('spread',F.col('ask')-F.col('bid'))
            .withColumn('relative_spread',F.col('spread')/F.col('mid_price'))
            .withColumn('spread_bps',F.col('relative_spread')*10000)
            .withColumn('has_volume',F.col('ask_volume').isNotNull() & F.col('bid_volume').isNotNull())
            .select('timestamp_utc','symbol',*fields,'mid_price','spread','relative_spread','spread_bps',
                    'source_id','source_file_id','source_row_id','tick_sequence','quality_flag','has_volume',
                    'is_reconstructed','year','month','day','matching_sources'))

say('Un désaccord entre canaux a un indicateur CONFLICT. Le dataset d’apprentissage peut exclure ces observations par configuration.')

## 08 · Silver Clean Ticks

Les données restent décimales, avec 12 décimales pour les cotations et les volumes. Les prix sont en USD par once troy, le spread relatif est sans unité et spread_bps est en points de base. Le rang est déterministe et l’ordre physique est connu à l’intérieur d’un fichier. Les volumes sont des volumes de cotation. Les identifiants relient chaque ligne à son fichier brut.

| Colonne | Type et contrainte |
|---|---|
| timestamp_utc | TIMESTAMP UTC à la milliseconde, obligatoire |
| symbol | STRING, XAUUSD |
| ask et bid | DECIMAL(24,12), USD/once, Ask ≥ Bid > 0 |
| ask_volume et bid_volume | DECIMAL(24,12) nullable, volume de cotation non négatif dans l’unité documentée |
| mid_price | DECIMAL(26,13), USD/once, moyenne du Bid et de l’Ask |
| spread, relative_spread et spread_bps | DECIMAL, USD/once, ratio et points de base respectivement |
| source_id, source_file_id et source_row_id | STRING, origine retenue, fichier relatif et identifiant physique stable |
| tick_sequence | INT, rang positif au sein d’un horodatage |
| quality_flag | STRING, OK, CONFLICT ou PRICE_ONLY selon la provenance et les conflits |
| has_volume et is_reconstructed | BOOLEAN, les deux volumes présents et reconstruction prouvée respectivement |
| year, month et day | INT, date UTC dérivée |
| matching_sources | ARRAY de STRING, canaux appariant la même occurrence |


In [ ]:
def chunk_frames(base, prefix):
    paths=[base/f'chunk={a}' for a,_ in month_windows(start,end) if completed(f'{prefix}_{a}')]
    if not paths:
        raise FileNotFoundError(f'Aucun checkpoint {prefix}')
    return spark.read.option('recursiveFileLookup','true').parquet(*map(str,paths))

def read_ticks():
    paths=[]
    for a,_ in month_windows(start,end):
        record=manifest(f'silver_{a}')
        if record.get('run_key')!=run_key or record.get('status')!='complete':
            raise RuntimeError(f'Silver incomplet : {a}')
        if record['silver']:
            part=TICKS/f'year={a.year}'/f'month={a.month:02d}'
            if not (part/'_SUCCESS').exists() or read_json(part/'_pipeline.json',{}).get('run_key')!=run_key:
                raise ValueError(f'Marqueur de partition Silver absent ou incompatible : {part}')
            paths.append(str(part))
    if not paths:
        raise RuntimeError('Silver sans tick éligible')
    return spark.read.option('basePath',str(TICKS)).parquet(*paths)

if active('08'):
    for a,b in month_windows(start,end):
        name=f'silver_{a}'
        if completed(name):
            continue
        df=require(f'normalized_{a}',NORM/f'chunk={a}')
        stats=df.agg(F.count('*').alias('raw'),
             F.count_if(F.col('reason').isNull()).alias('accepted'),
             F.count_if(F.col('reason')=='confirmed_historical_overlap').alias('historical_overlap'),
             F.count_if(F.col('reason').isNotNull() & (F.col('reason')!='confirmed_historical_overlap')).alias('rejected')).first().asDict()
        assert stats['raw']==sum(stats[k] for k in ('accepted','historical_overlap','rejected'))
        if stats['rejected']:
            publish(df.where(F.col('reason').isNotNull() & (F.col('reason')!='confirmed_historical_overlap')),
                    QUAR/f'chunk={a}')
        result_count=0
        rebuilt=0
        gaps=0
        if stats['accepted']:
            # L’ordre de lecture est préservé dans les identifiants avant le repartitionnement Spark.
            silver=reconcile(df.where('reason IS NULL')).persist(StorageLevel.DISK_ONLY)
            counts=silver.agg(F.count('*').alias('ticks'),F.count_if('is_reconstructed').alias('rebuilt'),
                  F.count_if(F.col('quality_flag')!='OK').alias('conflicts')).first()
            result_count,rebuilt,gaps=counts.ticks,counts.rebuilt,counts.conflicts
            part=TICKS/f'year={a.year}'/f'month={a.month:02d}'
            publish(silver.drop('year','month'),part)
            silver.unpersist()
        finish(name,period=[str(a),str(b)],**stats,silver=result_count,reconstructed=rebuilt,
               duplicate_copies=stats['accepted']-result_count,conflicts=gaps)
    by_month=[manifest(f'silver_{a}') for a,_ in month_windows(start,end)]
    finish('silver',**{k:sum(row[k] for row in by_month) for k in
        ('raw','accepted','historical_overlap','rejected','silver','reconstructed','duplicate_copies','conflicts')})
else:
    status('silver')
say('Silver est stocké en Parquet ZSTD, sous year et month. Les lignes rejetées gardent leur texte et leur motif en quarantaine.')

## 09 · Validation Silver et profil mesuré

La réconciliation compare les entrées, les rejets et les occurrences dédupliquées. Les résultats Bronze antérieurs constituent une référence indépendante. Les analyses par minute ou par source s’écrivent sur disque.

In [ ]:
def scoped_reference(a,b,y):
    if a<=date(y,1,1) and b>=date(y+1,1,1):
        return next((int(row['ticks_locaux']) for row in annual if int(row['annee'])==y),None)
    return None

PROFILE = reports/'profil_spark'
if active('09'):
    silver=read_ticks()
    input_records=chunk_frames(NORM,'normalized')
    # Profil sur les entrées admissibles et rejetées, avec regroupement des motifs.
    profile=input_records.groupBy('source_id','year','reason').agg(F.count('*').alias('rows'))
    publish(profile,PROFILE/'anomalies')
    publish(span_profile(input_records),PROFILE/'coverage')
    publish(ordered_checks(input_records.where('timestamp_utc IS NOT NULL')),PROFILE/'order')
    publish(input_records.where('timestamp_utc IS NOT NULL').groupBy(
        'source_id',F.date_trunc('minute','timestamp_utc').alias('minute')).count(),
        PROFILE/'minute_activity')
    publish(input_records.groupBy('source_id','components').count(),PROFILE/'widths')
    publish(input_records.groupBy('source_id','year','month').agg(
        *[F.count_if(F.col(name).isNull()).alias(f'null_{name}') for name in ('timestamp_utc',*fields)],
        F.count_if(F.col('reason')=='negative_spread').alias('negative_spreads'),
        F.count_if(F.col('reason')=='negative_volume').alias('negative_volumes')),
        PROFILE/'nulls')
    eligible=input_records.where('reason IS NULL')
    publish(eligible.groupBy('source_id').agg(
        (F.count('*')-F.countDistinct(F.struct(*IDENTITY))).alias('exact_repetitions')),
        PROFILE/'duplicates')
    publish(eligible.groupBy('source_id').agg(
        F.count_if(F.expr('ask IS NULL OR bid IS NULL')).alias('null_prices'),
        F.count_if(F.expr('ask_volume IS NULL OR bid_volume IS NULL')).alias('missing_volume'),
        F.count_if(F.expr('ask=bid')).alias('zero_spread'),
        F.expr("percentile_approx(ask, array(0.01,0.5,0.99), 10000)").alias('ask_quantiles'),
        F.expr("percentile_approx(bid, array(0.01,0.5,0.99), 10000)").alias('bid_quantiles'),
        F.expr("percentile_approx(ask_volume, array(0.01,0.5,0.99), 10000)").alias('ask_volume_quantiles'),
        F.expr("percentile_approx(bid_volume, array(0.01,0.5,0.99), 10000)").alias('bid_volume_quantiles'),
        F.expr("percentile_approx(ask-bid, array(0.01,0.5,0.99), 10000)").alias('spread_quantiles')),
        PROFILE/'distributions')
    silver.createOrReplaceTempView('silver_ticks')
    result=spark.sql("""SELECT count(*) AS ticks, count_if(NOT has_volume) AS volume_missing,
        count_if(quality_flag='OK' AND NOT arrays_overlap(matching_sources,
                 array('historical_old','historical_recent')) AND source_id!='fixture') AS additional_without_historical_match,
        count_if(ask IS NULL OR bid IS NULL OR ask<=0 OR bid<=0 OR ask<bid OR ask_volume<0 OR bid_volume<0 OR
                 source_row_id IS NULL OR source_file_id IS NULL OR timestamp_utc IS NULL) AS invalid,
        min(timestamp_utc) AS first_utc,max(timestamp_utc) AS last_utc FROM silver_ticks""").first().asDict()
    stats=manifest('silver')
    assert result['ticks']==stats['silver'] and result['invalid']==0
    fixture_rows=silver.where("source_id = 'fixture'").limit(1).count()
    assert bool(fixture_rows)==(run_mode=='TEST')
    assert stats['accepted']==stats['silver']+stats['duplicate_copies']
    if run_mode=='FULL':
        assert stats['historical_overlap']==43478752
    if start<=date(2026,11,23)<end and 'jforex' in CFG['source_ids']:
        partial=input_records.where("source_id = 'jforex' AND timestamp_utc >= TIMESTAMP '2026-11-23' AND reason IS NULL")
        day=partial.agg(F.count('*').alias('ticks'),F.countDistinct(F.hour('timestamp_utc')).alias('hours')).first()
        if day.ticks:
            result['jforex_2026_11_23']={'ticks':day.ticks,'observed_hours_utc':day.hours,'status':'PARTIAL'}
    conflicts=silver.where("quality_flag = 'CONFLICT'").groupBy('timestamp_utc').agg(
        F.countDistinct(F.struct('ask','bid')).alias('price_variants'),
        F.countDistinct(F.struct('ask_volume','bid_volume')).alias('volume_variants'))
    publish(conflicts,PROFILE/'conflicts')
    if run_mode=='PILOT' and start==date(2025,8,5) and end==date(2025,8,6) and {'jforex','touhid'}<=set(CFG['source_ids']):
        quotes=input_records.where('reason IS NULL')
        direct=quotes.where("source_id='jforex'").select(*IDENTITY)
        mirror=quotes.where("source_id='touhid'").select(*IDENTITY)
        assert direct.count()==mirror.count()==156842
        assert not direct.exceptAll(mirror).take(1) and not mirror.exceptAll(direct).take(1)
        result['jforex_touhid_exact_multiset']=156842
    if run_mode=='PILOT' and start==date(2021,1,5) and end==date(2021,1,6) and 'historical_recent' in CFG['source_ids']:
        assert stats['reconstructed']==len(proofs)==121
        result['decimal_repairs_exact']=stats['reconstructed']
    hist=input_records.where("reason IS NULL AND source_id IN ('historical_old','historical_recent')")
    local_yearly=hist.groupBy('year').agg(F.count('*').alias('historical_accepted_in_window'))
    yearly=(silver.groupBy('year').count().join(local_yearly,'year','left')
            .fillna(0,subset=['historical_accepted_in_window']).orderBy('year'))
    rows=[row.asDict() | {'historical_reference':scoped_reference(start,end,row.year)}
          for row in yearly.limit(100).toLocalIterator()]
    put_json(reports/'silver_annuel.json',rows)
    finish('validation',**result)
else:
    status('validation')
say('Les comparaisons annuelles n’affichent la référence Bronze entière que lorsque l’année est intégralement dans la fenêtre.')

## 10 · Bougies OHLC multi-timeframe

Calcul direct depuis les ticks pour chaque horizon. Les durées et rendements sont calculés entre ticks d’une même bougie. L’ouverture et la clôture utilisent le couple horodatage/rang. La médiane du spread est approximative. Le calendrier de marché est laissé indéterminé faute d’une référence fournisseur complète.

In [ ]:
FREQ={'M1':60,'M5':300,'M15':900,'M30':1800,'H1':3600,'H4':14400,'D1':86400}
METRICS=['bid','ask','mid_price']

def bar_frame(ticks,tf,a,b):
    time=F.col('timestamp_utc')
    if tf in FREQ:
        slot=F.window(time,f'{FREQ[tf]} seconds')
        expanded=ticks.withColumn('slot',slot).withColumn('bar_open_time',F.col('slot.start')).withColumn('bar_close_time',F.col('slot.end'))
    else:
        opening=F.date_trunc('week' if tf=='W1' else 'month',time)
        closing=(opening+F.expr('INTERVAL 7 DAYS')) if tf=='W1' else F.add_months(opening,1).cast('timestamp')
        expanded=ticks.withColumn('bar_open_time',opening).withColumn('bar_close_time',closing)
    order=F.struct('timestamp_utc','tick_sequence')
    w=Window.partitionBy('symbol','bar_open_time').orderBy('timestamp_utc','tick_sequence')
    expanded=(expanded.withColumn('previous_mid',F.lag('mid_price').over(w))
              .withColumn('delta_ms',F.unix_millis(time)-F.unix_millis(F.lag(time).over(w)))
              .withColumn('log_return',F.log(F.col('mid_price').cast('double')/F.col('previous_mid').cast('double'))))
    metrics=[]
    for name in METRICS:
        col='mid' if name=='mid_price' else name
        metrics.extend([F.min_by(name,order).alias(f'{col}_open'),F.max(name).alias(f'{col}_high'),
                        F.min(name).alias(f'{col}_low'),F.max_by(name,order).alias(f'{col}_close')])
    for name in ('ask_volume','bid_volume'):
        metrics.extend(fn(F.col(name)).alias(f'{name}_{suffix}') for suffix,fn in
                       [('sum',F.sum),('mean',F.avg),('min',F.min),('max',F.max),('std',F.stddev_pop)])
    bars=expanded.groupBy('symbol','bar_open_time','bar_close_time').agg(
        *metrics,F.count('*').alias('tick_count'),F.count_if(F.col('mid_price')>F.col('previous_mid')).alias('uptick_count'),
        F.count_if(F.col('mid_price')<F.col('previous_mid')).alias('downtick_count'),
        F.count_if(F.col('mid_price')==F.col('previous_mid')).alias('unchanged_tick_count'),
        (F.avg('delta_ms')/1000).alias('mean_inter_tick_duration'),(F.max('delta_ms')/1000).alias('max_inter_tick_duration'),
        F.count_if('has_volume').alias('volume_ticks'),F.min_by('spread',order).alias('spread_open'),F.max_by('spread',order).alias('spread_close'),
        F.min('spread').alias('spread_min'),F.max('spread').alias('spread_max'),F.avg('spread').alias('spread_mean'),
        F.percentile_approx('spread',.5,10000).alias('spread_median'),F.stddev_pop('spread').alias('spread_std'),
        F.avg('mid_price').alias('mid_mean'),F.var_pop('mid_price').alias('mid_variance'),F.stddev_pop('mid_price').alias('mid_std'),
        F.sum(F.pow('log_return',2)).alias('realized_variance'),
        F.sum(F.when(F.col('has_volume'),F.col('bid_volume')-F.col('ask_volume'))).alias('imbalance_num'),
        F.sum(F.when(F.col('has_volume'),F.col('bid_volume')+F.col('ask_volume'))).alias('imbalance_den'),
        F.sort_array(F.array_distinct(F.flatten(F.collect_set('matching_sources')))).alias('source_coverage'),
        F.count_if(F.col('quality_flag')!='OK').alias('conflict_ticks'),
        F.min('timestamp_utc').alias('first_tick'),F.max('timestamp_utc').alias('last_tick'))
    return (bars.withColumn('timeframe',F.lit(tf))
        .withColumn('bar_seconds',(F.unix_millis('bar_close_time')-F.unix_millis('bar_open_time'))/1000)
        .withColumn('tick_frequency',F.col('tick_count')/F.col('bar_seconds'))
        .withColumn('volume_coverage_ratio',F.col('volume_ticks')/F.col('tick_count'))
        .withColumn('quote_imbalance',F.try_divide('imbalance_num','imbalance_den'))
        .withColumn('realized_volatility',F.sqrt('realized_variance'))
        .withColumn('available_at',F.col('bar_close_time'))
        .withColumn('is_partial',(F.col('bar_open_time')<F.lit(str(a)).cast('timestamp')) |
                                  (F.col('bar_close_time')>F.lit(str(b)).cast('timestamp')))
        .withColumn('has_missing_data',F.coalesce(F.col('max_inter_tick_duration')>CFG['max_gap_seconds'],F.lit(False)))
        .withColumn('expected_session_status',F.lit('UNKNOWN'))
        .withColumn('quality_flag',F.when(F.col('is_partial'),'PARTIAL').when(F.col('conflict_ticks')>0,'CONFLICT')
                     .when(F.col('has_missing_data'),'GAP').otherwise('OK'))
        .withColumn('year',F.year('bar_open_time')).withColumn('month',F.month('bar_open_time'))
        .drop('imbalance_num','imbalance_den'))

if active('10'):
    ticks=read_ticks()
    counts={}
    for tf in CFG['timeframes']:
        if not completed(f'bars_{tf}'):
            frame=bar_frame(ticks,tf,start,end)
            publish(frame,BARS/tf,('year','month'))
            stored=spark.read.parquet(str(BARS/tf))
            r=stored.agg(F.count('*').alias('bars'),F.sum('tick_count').alias('ticks')).first()
            assert r.ticks==manifest('silver')['silver']
            finish(f'bars_{tf}',bars=r.bars,ticks=r.ticks)
        counts[tf]=manifest(f'bars_{tf}')['bars']
    finish('bars',counts=counts)
else:
    status('bars')
say('Une bougie n’existe que si des ticks existent. Une période ou un gap non confirmé reste signalé.')

## 11 · Indicateurs Gold obligatoires

EMA et RMA de Wilder démarrent avec la moyenne des premières valeurs nécessaires. RSI vaut 50 si les gains et pertes sont tous deux nuls. Le Stochastique utilise %K brut et le lissage simple %D. Bollinger utilise l’écart-type de population. Tick-OBV s’appuie sur le nombre de cotations.

In [ ]:
from statistics import fmean, pstdev

def average(values,n):
    values=list(values)[-n:]
    return fmean(values) if len(values)==n and all(value is not None for value in values) else None

def divide(a,b):
    return a/b if a is not None and b is not None and b!=0 else None

class Average:
    def __init__(self,n,alpha):
        self.n,self.alpha,self.total,self.size,self.value=n,alpha,0.0,0,None
    def push(self,x):
        if x is None:
            return None
        if self.value is None:
            self.total+=x
            self.size+=1
            if self.size==self.n:
                self.value=self.total/self.n
        else:
            self.value+=self.alpha*(x-self.value)
        return self.value

class Indicators:
    def __init__(self,cfg):
        self.cfg=cfg
        macd_periods={n for short,long,_ in cfg['macd'] for n in (short,long)}
        self.maxlen=max([cfg['extra_window']+2,cfg['roc']+2,*cfg['sma'],*cfg['cci'],
                        *[p for pair in cfg['stoch']+cfg['bollinger'] for p in pair if isinstance(p,int)]])+2
        self.h=deque(maxlen=self.maxlen)
        self.log_returns=deque(maxlen=self.maxlen)
        self.obvs=deque(maxlen=max(cfg['obv'])+1)
        self.emas={n:Average(n,2/(n+1)) for n in set(cfg['ema']) | macd_periods}
        self.signals={p:Average(p[2],2/(p[2]+1)) for p in cfg['macd']}
        self.rsi={n:(Average(n,1/n),Average(n,1/n)) for n in cfg['rsi']}
        self.atr={n:Average(n,1/n) for n in cfg['atr']}
        self.stoch={pair:deque(maxlen=pair[1]) for pair in cfg['stoch']}
        n=cfg['adx']
        self.adx=[Average(n,1/n) for _ in range(4)]
        self.sar=None
        self.obv=0.0
        self.last_micro=None

    def update(self,row):
        c=float(row['mid_close'])
        o,h,l=(float(row[k]) for k in ('mid_open','mid_high','mid_low'))
        prev=self.h[-1] if self.h else None
        typical=(h+l+c)/3
        change=c-prev[3] if prev else None
        tr=max(h-l,abs(h-prev[3]),abs(l-prev[3])) if prev else h-l
        self.h.append((o,h,l,c,typical))
        self.log_returns.append(math.log(c/prev[3]) if prev else None)
        history=list(self.h)
        prices=[x[3] for x in history]
        result={f'sma_{n}':average(prices,n) for n in self.cfg['sma']}
        ema={n:smoother.push(c) for n,smoother in self.emas.items()}
        result.update({f'ema_{n}':ema[n] for n in self.cfg['ema']})
        for n,(gain,loss) in self.rsi.items():
            up=gain.push(max(0,change) if change is not None else None)
            down=loss.push(max(0,-change) if change is not None else None)
            result[f'rsi_{n}']=None if up is None or down is None else (
                (100 if up else 50) if down==0 else 100-100/(1+up/down))
        for triple,state in self.signals.items():
            short,long,signal=triple
            suffix=f'{short}_{long}_{signal}'
            line=ema[short]-ema[long] if ema[short] is not None and ema[long] is not None else None
            smooth=state.push(line)
            result.update({f'macd_{suffix}':line,f'macd_signal_{suffix}':smooth,
                           f'macd_hist_{suffix}':line-smooth if smooth is not None else None})
        for (n,d),buffer in self.stoch.items():
            window=history[-n:]
            low,high=min(x[2] for x in window),max(x[1] for x in window)
            k=divide(100*(c-low),high-low) if len(window)==n else None
            buffer.append(k)
            result.update({f'stoch_k_{n}_{d}':k,f'stoch_d_{n}_{d}':average(buffer,d)})
        for n in self.cfg['cci']:
            window=[x[4] for x in history[-n:]]
            mean=average(window,n)
            result[f'cci_{n}']=divide(typical-mean,.015*fmean(abs(x-mean) for x in window)) if mean is not None else None
        for n,state in self.atr.items():
            result[f'atr_{n}']=state.push(tr)
        for n,multiple in self.cfg['bollinger']:
            middle=average(prices,n)
            radius=multiple*pstdev(prices[-n:]) if middle is not None else None
            lower,upper=(middle-radius,middle+radius) if radius is not None else (None,None)
            k=f'{n}_{multiple:g}'
            result.update({f'bb_middle_{k}':middle,f'bb_upper_{k}':upper,f'bb_lower_{k}':lower,
                           f'bb_bandwidth_{k}':divide(2*radius,middle) if radius is not None else None,
                           f'bb_percent_b_{k}':divide(c-lower,2*radius) if radius is not None else None})
        self.obv += ((change>0)-(change<0))*row['tick_count'] if change is not None else 0
        self.obvs.append(self.obv)
        result['tick_obv']=self.obv
        result.update({f'tick_obv_delta_{lag}':self.obv-self.obvs[-lag-1] if len(self.obvs)>lag else None
                       for lag in self.cfg['obv']})
        result.update(extra_indicators(self,row,history,prev,tr,result))
        return result

say('Les formules obligatoires utilisent exclusivement les bougies déjà reçues dans l’ordre.')

## 12 · Features complémentaires

Les familles s’activent dans `CFG['extra']`. Les statistiques de volatilité de Parkinson, Garman–Klass et Rogers–Satchell sont des estimateurs par bougie. Les sessions sont des plages UTC descriptives. Le SAR utilise une initialisation sur deux barres et borne la valeur aux extrêmes précédents.

Activer la famille `microstructure_variations` pour les variations du spread et des volumes de cotation.

In [ ]:
def sar_update(state,history):
    if len(history)<2:
        return None
    cfg=state.cfg
    o,h,l,c,_=history[-1]
    prev=history[-2]
    step,maximum=cfg['sar_step'],cfg['sar_max']
    if state.sar is None:
        up=c>=prev[3]
        state.sar=(up,min(l,prev[2]) if up else max(h,prev[1]),max(h,prev[1]) if up else min(l,prev[2]),step)
        return state.sar[1]
    up,sar,extreme,alpha=state.sar
    proposal=sar+alpha*(extreme-sar)
    recent=history[-3:-1]
    proposal=min(proposal,*(x[2] for x in recent)) if up else max(proposal,*(x[1] for x in recent))
    if (up and l<proposal) or (not up and h>proposal):
        state.sar=(not up,extreme,l if up else h,step)
    else:
        new_extreme=max(extreme,h) if up else min(extreme,l)
        state.sar=(up,proposal,new_extreme,min(maximum,alpha+step) if new_extreme!=extreme else alpha)
    return state.sar[1]

def quantile(values,q):
    x=sorted(values)
    at=(len(x)-1)*q
    lo=int(at)
    return x[lo]+(x[min(lo+1,len(x)-1)]-x[lo])*(at-lo)

def extra_indicators(state,row,history,prev,tr,required):
    cfg=state.cfg
    enabled=cfg['extra']
    n,k=cfg['extra_window'],cfg['roc']
    o,h,l,c,_=history[-1]
    p=prev[3] if prev else None
    prices=[x[3] for x in history]
    sample=prices[-n:]
    window=history[-n:]
    complete=len(sample)==n
    previous=prices[-k-1] if len(prices)>k else None
    r=state.log_returns[-1]
    out={}
    if 'price' in enabled:
        out.update(simple_return=c/p-1 if p else None,log_return=r,lagged_log_return=state.log_returns[-2] if len(state.log_returns)>1 else None,
                   momentum=c-previous if previous else None,price_range=h-l,candle_body=c-o,upper_wick=h-max(o,c),
                   lower_wick=min(o,c)-l,high_low_ratio=h/l,close_open_ratio=c/o)
        for p in cfg['sma']:
            avg=required[f'sma_{p}']
            out[f'distance_sma_{p}']=c/avg-1 if avg else None
    if 'trend' in enabled:
        out[f'wma_{n}']=sum((i+1)*x for i,x in enumerate(sample))/(n*(n+1)/2) if complete else None
        plus,minus,tavg,davg=state.adx
        u,d=(h-prev[1],prev[2]-l) if prev else (0.0,0.0)
        pr=plus.push(u if u>d and u>0 else 0)
        mr=minus.push(d if d>u and d>0 else 0)
        den=tavg.push(tr)
        plus_di,minus_di=divide(100*pr,den) if pr is not None else None,divide(100*mr,den) if mr is not None else None
        dx=(divide(100*abs(plus_di-minus_di),plus_di+minus_di) or 0.0) if plus_di is not None and minus_di is not None else None
        out.update(dmi_plus=plus_di,dmi_minus=minus_di,adx=davg.push(dx),parabolic_sar=sar_update(state,history))
    if 'momentum' in enabled:
        high=max(x[1] for x in window)
        low=min(x[2] for x in window)
        out.update(roc=100*(c/previous-1) if previous else None,
                   williams_r=divide(-100*(high-c),high-low) if complete else None)
    if 'volatility' in enabled:
        hl=math.log(h/l)
        co=math.log(c/o)
        rs=math.log(h/o)*math.log(h/c)+math.log(l/o)*math.log(l/c)
        out.update(parkinson_variance=hl**2/(4*math.log(2)),garman_klass_variance=.5*hl**2-(2*math.log(2)-1)*co**2,
                   rogers_satchell_variance=rs,realized_variance=float(row['realized_variance']) if row['realized_variance'] is not None else None,
                   realized_volatility=float(row['realized_volatility']) if row['realized_volatility'] is not None else None)
        for p in cfg['atr']:
            out[f'relative_atr_{p}']=divide(required[f'atr_{p}'],c)
    if 'statistics' in enabled:
        mean=fmean(sample) if complete else None
        sd=pstdev(sample) if complete else None
        z=[(x-mean)/sd for x in sample] if sd else []
        lag=divide(sum((a-mean)*(b-mean) for a,b in zip(sample,sample[1:])),sum((x-mean)**2 for x in sample)) if sd else None
        out.update(rolling_mean=mean,rolling_std=sd,rolling_variance=sd**2 if sd is not None else None,
                   z_score=divide(c-mean,sd) if mean is not None else None,skewness=fmean(t**3 for t in z) if z else None,
                   kurtosis=fmean(t**4 for t in z)-3 if z else None,autocorr_1=lag,
                   rolling_q10=quantile(sample,.1) if complete else None,rolling_q90=quantile(sample,.9) if complete else None,
                   rolling_min=min(sample) if complete else None,rolling_max=max(sample) if complete else None,
                   rolling_drawdown=c/max(sample)-1 if complete else None)
    if 'microstructure' in enabled:
        spread=float(row['spread_close'])
        out.update(spread_abs=spread,spread_rel=spread/c,spread_bps=10000*spread/c,
                   spread_volatility=row['spread_std'],tick_frequency=row['tick_frequency'],
                   tick_imbalance=divide(row['uptick_count']-row['downtick_count'],row['tick_count']),
                   volume_imbalance=row['quote_imbalance'],intertick_seconds=row['mean_inter_tick_duration'])
    if 'microstructure_variations' in enabled:
        v=[row.get('ask_volume_sum'),row.get('bid_volume_sum')]
        current=(float(row['spread_close']),sum(map(float,v)) if all(x is not None for x in v) else None)
        last=state.last_micro
        out.update(spread_change=current[0]-last[0] if last else None,
                   quote_volume_change=current[1]-last[1] if last and None not in (current[1],last[1]) else None)
        state.last_micro=current
    if 'time' in enabled:
        stamp=row['bar_open_time']
        h=stamp.hour+(stamp.minute/60)
        asia,europe,us=h<9,7<=h<16,13<=h<22
        out.update(hour=h,weekday=stamp.weekday(),month_of_year=stamp.month,asia=int(asia),europe=int(europe),
                   us=int(us),overlapping_sessions=int(sum((asia,europe,us))>1),
                   hour_sin=math.sin(2*math.pi*h/24),hour_cos=math.cos(2*math.pi*h/24))
    return out

say('Les familles complémentaires calculent des valeurs descriptives à partir des prix et de l’activité cotée.')

## 13 · Features multi-timeframe

La jointure utilise des intervalles de disponibilité semi-ouverts. À égalité, la nouvelle bougie clôturée est disponible. Une tranche par jour UTC borne les jointures et les mesures de continuité.

Une valeur de fréquence supérieure expire à la fin de la période attendue si la bougie suivante manque.

In [ ]:
def with_higher(main,higher,tf,features,until):
    suffix='_'+tf
    good=higher.where(~F.col('is_partial'))
    w=Window.partitionBy('symbol').orderBy('available_at')
    close=F.col('available_at')
    if tf in FREQ:
        expiry=close+F.expr(f'INTERVAL {FREQ[tf]} SECONDS')
    elif tf=='W1':
        expiry=close+F.expr('INTERVAL 7 DAYS')
    else:
        expiry=F.add_months(close,1).cast('timestamp')
    limit=F.lit(until).cast('timestamp')+F.expr('INTERVAL 1 MICROSECOND')
    base=good.withColumn('next_time',F.least(F.coalesce(F.lead('available_at').over(w),limit),expiry))
    base=base.where('next_time > available_at').withColumn('join_date',F.explode(F.sequence(
        F.to_date('available_at'),F.to_date(F.col('next_time')-F.expr('INTERVAL 1 MICROSECOND')))))
    keep=[name for name in features if name in higher.columns]
    base=base.select('symbol','join_date',F.col('available_at').alias('lower'),F.col('next_time').alias('upper'),
        F.col('quality_flag').alias('quality'+suffix),*[F.col(x).alias(x+suffix) for x in keep])
    assigned=main.withColumn('join_date',F.to_date('available_at')).alias('m')
    other=base.alias('h')
    match=((F.col('m.symbol')==F.col('h.symbol')) & (F.col('m.join_date')==F.col('h.join_date')) &
           (F.col('m.available_at')>=F.col('h.lower')) & (F.col('m.available_at')<F.col('h.upper')))
    return (assigned.join(other,match,'left').select('m.*',
            F.col('h.lower').alias('available_at'+suffix), F.col('h.quality'+suffix),
            *[F.when(F.col('h.quality'+suffix).isin(*CFG['allowed_quality']),F.col('h.'+x+suffix)).alias(x+suffix) for x in keep])
            .drop('join_date'))

say('Une feature de fréquence supérieure conserve son horodatage de disponibilité et son statut de qualité.')

## 14 · État récursif et écriture Gold

Chaque timeframe est réparti par plages ordonnées. Les partitions d’agrégats sont lues en flux, avec un petit tampon Arrow et un état continu. Un changement de partition ou de mois ne réinitialise pas les indicateurs. Une interruption identifiée ou une bougie non admissible ouvre un nouveau segment.

In [ ]:
def demo_bar(index=0,price=10.0):
    start_time=datetime(2025,8,5)+timedelta(minutes=index)
    return dict(mid_close=price,mid_open=price,mid_high=price+1,mid_low=price-1,tick_count=3,
          bar_open_time=start_time,bar_close_time=start_time+timedelta(minutes=1),
          quality_flag='OK',spread_close=.1,spread_std=.01,tick_frequency=.1,uptick_count=2,
          downtick_count=0,quote_imbalance=.3,mean_inter_tick_duration=2.0,
          realized_variance=.03,realized_volatility=math.sqrt(.03))

FEATURES=list(Indicators(CFG).update(demo_bar()))
BAR_META=['symbol','bar_open_time','bar_close_time','available_at','ask_close','bid_close','mid_close',
           'mid_open','mid_high','mid_low','tick_count','volume_coverage_ratio','quality_flag','is_partial',
           'source_coverage','has_missing_data']
def feature_stream(rows,cfg):
    state=Indicators(cfg)
    previous=None
    segment=0
    for item in rows:
        row=item.asDict(recursive=False) if hasattr(item,'asDict') else dict(item)
        opening,closing=row['bar_open_time'],row['bar_close_time']
        if previous is not None and opening<previous:
            raise ValueError('Bougies en recul ou en chevauchement dans le flux ordonné')
        allowed=row['quality_flag'] in cfg['allowed_quality'] and not row.get('is_partial',False)
        if (previous is not None and opening!=previous) or not allowed:
            state=Indicators(cfg)
            segment+=1
        features=state.update(row) if allowed else dict.fromkeys(FEATURES)
        yield row | {'segment':segment} | features
        previous=closing

def gold_write(frame,path):
    import pyarrow as pa
    import pyarrow.parquet as pq
    import time
    os.environ['TZ']='UTC'
    if hasattr(time,'tzset'):
        time.tzset()
    marker=path/'_pipeline.json'
    if path.exists():
        state=read_json(marker,{})
        if state.get('run_key')==run_key:
            return state['rows']
        raise FileExistsError(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(f'.{path.name}.{uuid4().hex}.pending')
    temp.mkdir()
    stamps={'bar_open_time','bar_close_time','available_at'}
    prices={'ask_close','bid_close','mid_close','mid_open','mid_high','mid_low'}
    def arrow_type(key):
        if key in stamps:
            return pa.timestamp('us',tz='UTC')
        if key in prices:
            return pa.decimal128(26,13)
        if key in {'tick_count','segment'}:
            return pa.int64()
        if key in {'is_partial','has_missing_data'}:
            return pa.bool_()
        if key=='source_coverage':
            return pa.list_(pa.string())
        if key in {'symbol','quality_flag'}:
            return pa.string()
        return pa.float64()
    names=[*BAR_META,'segment',*FEATURES]
    schema=pa.schema([(key,arrow_type(key)) for key in names])
    floats={field.name for field in schema if pa.types.is_float64(field.type)}
    cached=[]
    part=0
    count=0
    ordered=frame.repartitionByRange(max(8,workers*8),'bar_open_time').sortWithinPartitions('bar_open_time')
    def flush():
        nonlocal part
        pq.write_table(pa.Table.from_pylist(cached,schema=schema),temp/f'part-{part:06d}.parquet',compression='zstd')
        cached.clear()
        part+=1
    for row in feature_stream(ordered.toLocalIterator(prefetchPartitions=False),CFG):
        cached.append({key:float(row[key]) if key in floats and row.get(key) is not None else row.get(key)
                       for key in names})
        count+=1
        if len(cached)>=4096:
            flush()
    if cached:
        flush()
    assert count>0
    put_json(temp/'_pipeline.json',{'run_key':run_key,'rows':count})
    temp.rename(path)
    return count

if active('14') or active('11') or active('12') or active('13'):
    for tf in CFG['timeframes']:
        if not completed(f'gold_{tf}'):
            bars=require(f'bars_{tf}',BARS/tf)
            n=gold_write(bars,GOLD/tf)
            assert n==manifest(f'bars_{tf}')['bars']
            finish(f'gold_{tf}',rows=n)
    finish('gold',features=len(FEATURES),timeframes=CFG['timeframes'])
else:
    status('gold')
say(f'{len(FEATURES)} colonnes de features configurées. Le tampon Arrow est limité à 4 096 bougies.')

## 15 · Prévention du data leakage

Les barres incomplètes restent consultables avec leur statut. Les fenêtres glissantes sont causales. Les intervalles `available_at` sont contrôlés après jointure. Le test modifie le futur et vérifie que le passé reste identique.

In [ ]:
def causal_reference_test():
    rows=[demo_bar(i,10+i/9) for i in range(85)]
    previous=[{k:r[k] for k in FEATURES} for r in feature_stream(rows,CFG)]
    changed=rows[:55]+[demo_bar(i,100+i) for i in range(55,85)]
    next_run=[{k:r[k] for k in FEATURES} for r in feature_stream(changed,CFG)]
    assert previous[:55]==next_run[:55]
    assert previous[-1]!=next_run[-1]
    # Le changement de limite d’itération ne modifie pas les valeurs récursives.
    from itertools import chain
    def sliced(rows,k):
        yield from rows[:k]
        yield from rows[k:]
    assert list(feature_stream(sliced(rows,23),CFG))==list(feature_stream(rows,CFG))
    gap=rows[:40]+rows[41:]
    result=list(feature_stream(gap,CFG))
    assert result[40]['segment']>result[39]['segment'] and result[40]['sma_5'] is None
    return True

if active('15'):
    assert causal_reference_test()
    finish('causality',mutated_future=True,gap_reset=True,partition_boundary=True)
else:
    status('causality')
say('Le test vérifie aussi que la récursion conserve son état lorsqu’une limite de lecture change.')

## 16 · Dataset Builder

Les splits se suivent dans le temps. Le scaler est ajusté sur Train et appliqué ensuite aux autres splits. Les prix d’exécution restent dans leurs colonnes décimales. La fenêtre d’observation est un descripteur consultable dans les barres Gold.

In [ ]:
def build_dataset():
    base=CFG['main']
    full=require(f'gold_{base}',GOLD/base).where(F.col('symbol')==CFG['instrument'])
    if CFG['dataset_start']:
        full=full.where(F.col('available_at')>=F.lit(CFG['dataset_start']).cast('timestamp'))
    if CFG['dataset_end']:
        full=full.where(F.col('available_at')<F.lit(CFG['dataset_end']).cast('timestamp'))
    wanted=CFG['features']
    for tf in CFG['higher']:
        if tf==base:
            continue
        secondary=require(f'gold_{tf}',GOLD/tf)
        requested=[key[:-len(tf)-1] for key in wanted if key.endswith('_'+tf)]
        if not requested:
            requested=[key for key in wanted if key in FEATURES]
        full=with_higher(full,secondary,tf,requested,str(end))
    if not set(wanted)<=set(full.columns):
        raise ValueError(f'Features absentes : {set(wanted)-set(full.columns)}')
    future=F.lit(False)
    for column in (name for name in full.columns if name.startswith('available_at_')):
        future=future | (F.col(column)>F.col('available_at'))
    assert not full.where(future).take(1)

    ok=(F.col('quality_flag').isin(*CFG['allowed_quality']) & ~F.col('is_partial') &
        F.forall('source_coverage',lambda item:item.isin(*CFG['dataset_sources'])))
    if CFG['volume_required']:
        ok=ok & (F.col('volume_coverage_ratio')==1)
    for name in wanted:
        ok=ok & F.col(name).isNotNull() & ~F.isnan(F.col(name).cast('double'))
    look=max(1,int(CFG['lookback']))
    permitted=full.withColumn('eligible',ok)
    if look>1:
        # Un segment est interrompu aux gaps par l’étape Gold. La fenêtre est calculée avant de filtrer.
        w=Window.partitionBy('symbol','segment').orderBy('available_at').rowsBetween(-look+1,0)
        permitted=permitted.withColumn('window_start',F.min('available_at').over(w)).where(
                  F.sum(F.col('eligible').cast('int')).over(w)==look)
    else:
        permitted=permitted.withColumn('window_start',F.col('available_at')).where('eligible')
    limite=permitted.agg(F.min('available_at'),F.max('available_at')).first()
    first,last=limite[0],limite[1]
    if first is None or last is None or first==last:
        say('**Dataset indisponible** · fenêtre trop courte ou warm-up insuffisant.')
        return None
    split=CFG['split']
    bounds=[first+(last-first)*p for p in (.70,.85)] if split is None else [datetime.fromisoformat(s) for s in split]
    assert len(bounds)==2 and bounds[0]<bounds[1]
    permitted=permitted.withColumn('split',F.when(F.col('available_at')<F.lit(bounds[0]),'train').when(
        F.col('available_at')<F.lit(bounds[1]),'validation').otherwise('test'))
    train=permitted.where(F.col('split')=='train')
    if not train.take(1):
        say('**Dataset indisponible** · Train est vide sur la fenêtre configurée.')
        return None
    scaler=CFG['scaler']
    assert scaler in {'standard','minmax','none'}
    numeric={}
    if scaler!='none':
        fn=(F.avg,F.stddev_pop) if scaler=='standard' else (F.min,F.max)
        values=train.agg(*[operation(name).cast('double').alias(f'{i}_{j}')
                           for i,name in enumerate(wanted) for j,operation in enumerate(fn)]).first()
        for i,name in enumerate(wanted):
            location,variance=values[f'{i}_0'],values[f'{i}_1']
            scale=(variance-location) if scaler=='minmax' and variance is not None else variance
            scale=scale or 1.0
            numeric[name]=[location,scale]
            permitted=permitted.withColumn('scaled_'+name,(F.col(name)-location)/scale)
    clocks=[name for name in full.columns if name.startswith('available_at_') or name.startswith('quality_') and name!='quality_flag']
    result=permitted.select('symbol','available_at','window_start','ask_close','bid_close','mid_close',
               'split','segment','quality_flag','source_coverage',*clocks,*wanted,
               *['scaled_'+name for name in numeric])
    publish(result,BUILT,('split',))
    checked=spark.read.parquet(str(BUILT))
    counts={r['split']:r['count'] for r in checked.groupBy('split').count().limit(3).toLocalIterator()}
    finish('dataset',rows=sum(counts.values()),splits=counts,cutoffs=[str(x) for x in bounds],features=wanted,
           scaler=scaler,scaler_train=numeric,lookback=look,window_reference=str((GOLD/base).relative_to(out)))
    return counts

if active('16') and not completed('dataset'):
    build_dataset()
else:
    status('dataset')
say('Les sorties contiennent l’instant de disponibilité, la fenêtre, les features et les prix Bid/Ask de référence.')

## 17 · Tests automatisés

Les assertions utilisent de petits cas indépendants. Ces données sont réservées au mode TEST et aux assertions. La fusion pilote réelle utilise les fichiers Bronze.

In [ ]:
def assert_close(actual,value):
    assert actual is not None and math.isclose(actual,value,rel_tol=1e-9,abs_tol=1e-9),(actual,value)

def test_indicators():
    rows=[demo_bar(i,10+i/2) for i in range(75)]
    result=list(feature_stream(rows,CFG))
    final=result[-1]
    for n in CFG['sma']:
        assert_close(final[f'sma_{n}'],47-(n-1)/4)
    for n in CFG['ema']:
        assert_close(final[f'ema_{n}'],47-(n-1)/4)
        assert result[n-2][f'ema_{n}'] is None
    for n in CFG['rsi']:
        assert_close(final[f'rsi_{n}'],100)
    for a,b,signal in CFG['macd']:
        suffix=f'{a}_{b}_{signal}'
        assert_close(final[f'macd_{suffix}'],(b-a)/4)
        assert_close(final[f'macd_signal_{suffix}'],(b-a)/4)
        assert_close(final[f'macd_hist_{suffix}'],0)
    for n,d in CFG['stoch']:
        known=100*(1+(n-1)*.5)/(2+(n-1)*.5)
        assert_close(final[f'stoch_k_{n}_{d}'],known)
        assert_close(final[f'stoch_d_{n}_{d}'],known)
    for n in CFG['cci']:
        mean=47-(n-1)/4
        md=sum(abs((47-i/2)-mean) for i in range(n))/n
        assert_close(final[f'cci_{n}'],(47-mean)/(.015*md))
    for n in CFG['atr']:
        assert_close(final[f'atr_{n}'],2)
    for n,k in CFG['bollinger']:
        middle=47-(n-1)/4
        radius=k*math.sqrt((n*n-1)/48)
        assert_close(final[f'bb_middle_{n}_{k:g}'],middle)
        assert_close(final[f'bb_upper_{n}_{k:g}'],middle+radius)
        assert_close(final[f'bb_lower_{n}_{k:g}'],middle-radius)
    for n in CFG['obv']:
        assert_close(final[f'tick_obv_delta_{n}'],3*n)
    assert final['tick_obv']==222
    assert causal_reference_test()

def test_spark():
    stamp='2025-08-05T01:00:00.000Z'
    def r(src,index,when,ask,bid,va='.002',vb='.003'):
        return (src,src+'.csv',index,None,when,ask,bid,va,vb)
    values=[r('touhid',0,stamp,'10','9'),r('touhid',1,stamp,'10','9'),
        r('jforex',0,stamp,'10','9'),r('jforex',1,stamp,'11','9'),
        r('jforex',2,'2025-08-05T01:03:00.000Z','12','11',None,None),
        r('jforex',3,'2025-08-05T01:05:00.000Z','13','12')]
    for i,clock in enumerate(['2025.01.05 03:00:00.000','2025.08.05 03:00:00.000',
                              '2025.03.30 03:30:00.000','2025.10.26 03:30:00.000']):
        line=f'{clock},14,5,14,3,0,01,0,02'
        values.append(('historical_recent','legacy_fixture.csv',i,line,None,None,None,None,None))
    decimal_line='2025.08.05 04:00:00.000,14,14,3,0,01,0,02'
    values.append(('historical_recent','legacy_fixture.csv',4,decimal_line,None,None,None,None,None))
    proof=[(sha256(decimal_line.encode()).hexdigest(),'2025-08-05T01:00:00Z',['14','14.3','.01','.02'])]
    raw=spark.createDataFrame(values,RAW_SCHEMA)
    check=normalize(clean(raw,proof),date(2025,1,1),date(2026,1,1)).persist()
    hist=check.where(F.col('source_id')=='historical_recent').orderBy('source_offset').take(5)
    assert [row.timestamp_utc.hour for row in hist[:2]]==[1,0]
    assert all(row.reason=='ambiguous_or_nonexistent_local_clock' for row in hist[2:4])
    assert hist[4].reason=='negative_spread' and hist[4].is_reconstructed
    strict=check.where("reason IS NULL AND source_id IN ('touhid','jforex')")
    merged=reconcile(strict).persist()
    assert merged.count()==5
    assert merged.where("timestamp_utc = TIMESTAMP '2025-08-05 01:00:00' AND ask=10").count()==2
    assert merged.where('NOT has_volume').count()==1
    assert merged.where("quality_flag='CONFLICT'").count()==3
    m5=bar_frame(merged,'M5',date(2025,8,5),date(2025,8,6))
    m15=bar_frame(merged,'M15',date(2025,8,5),date(2025,8,6))
    first=m5.orderBy('bar_open_time').first()
    assert (int(first.ask_open),int(first.ask_close),int(first.ask_high),first.tick_count)==(10,12,12,4)
    assert m15.first().tick_count==5
    join=with_higher(m5.withColumn('selected',F.lit(1.0)),m15.withColumn('selected',F.lit(8.0)),
                     'M15',['selected'],'2025-08-06')
    assert not join.where('available_at_M15>available_at').take(1)
    assert all(row.selected_M15 is None for row in join.take(2))
    # Même entrée avec une autre distribution physique.
    rerun=reconcile(strict.repartition(3))
    assert not merged.exceptAll(rerun).take(1)
    assert not rerun.exceptAll(merged).take(1)
    merged.unpersist()
    check.unpersist()
    with TemporaryDirectory(prefix='spark_idempotence_',dir=CFG['scratch']) as folder:
        p=Path(folder)/'checkpoint'
        publish(spark.range(3),p)
        publish(spark.range(3),p)
        assert spark.read.parquet(str(p)).count()==3
    return True

if active('17'):
    test_indicators()
    assert test_spark()
    finish('tests',passed=['decimals','UTC','DST','multiplicity','multisource','distinct_quotes','OHLC','timeframes',
        'SMA','EMA','RSI','MACD','stochastic','CCI','ATR','Bollinger','Tick-OBV','NULL','continuity','leakage',
        'idempotence','reproducibility'])
else:
    status('tests')
say('Les tests de reproductibilité comparent les lignes dans les deux sens après un repartitionnement.')

## 18 · Graphiques et supervision

La couverture utilise des agrégats. Les courbes d’indicateurs se limitent aux 240 dernières bougies de la fenêtre de démonstration. Le tableau de corrélations est calculé dans Spark sur un périmètre défini.

In [ ]:
def show_figures():
    import matplotlib.pyplot as plt
    from math import log10
    if annual:
        years=[int(x['annee']) for x in annual]
        base=[int(x['ticks_locaux'])/1e6 for x in annual]
        complement=[int(x['ticks_archive_independante'] or x['ticks_archive_carlos_silva'] or 0)/1e6 for x in annual]
        fig,ax=plt.subplots(figsize=(11,3))
        ax.bar([y-.19 for y in years],base,.38,label='Bronze historique')
        measured={row['year']:row['count']/1e6 for row in read_json(reports/'silver_annuel.json',[])} if run_mode=='FULL' else {}
        ax.bar([y+.19 for y in years],[measured.get(y,0) for y in years] if measured else complement,.38,
               label='Silver sur la fenêtre exécutée' if measured else 'Archives UTC auditées')
        ax.set(ylabel='Millions de ticks',xticks=years)
        ax.tick_params(axis='x',rotation=65)
        ax.legend()
        display(fig)
        plt.close(fig)
    if not spark or not completed('validation'):
        return
    bars=spark.read.parquet(str(PROFILE/'coverage')).groupBy('year','month').agg(F.sum('ticks').alias('ticks'))
    coverage=list(bars.limit(500).toLocalIterator())
    if coverage:
        years=sorted({r.year for r in coverage})
        mapping={(r.year,r.month):log10(1+r.ticks) for r in coverage}
        fig,ax=plt.subplots(figsize=(8,max(2,len(years)*.3)))
        image=ax.imshow([[mapping.get((y,m),0) for m in range(1,13)] for y in years],aspect='auto')
        ax.set(yticks=range(len(years)),yticklabels=years,xticks=range(12),xticklabels=range(1,13),title='Couverture mensuelle Spark')
        fig.colorbar(image,label='log10(ticks + 1)')
        display(fig)
        plt.close(fig)
    anomaly=list(spark.read.parquet(str(PROFILE/'anomalies')).where('reason IS NOT NULL').groupBy('reason').agg(F.sum('rows').alias('n')).limit(24).toLocalIterator())
    if anomaly:
        fig,ax=plt.subplots(figsize=(9,3))
        ax.barh([r.reason for r in anomaly],[r.n for r in anomaly])
        ax.set_title('Anomalies et exclusions')
        display(fig)
        plt.close(fig)
    sample=read_ticks().select('ask_volume','bid_volume','spread').sample(.08 if run_mode=='PILOT' else .0001,seed=17).limit(2000)
    vals=list(sample.toLocalIterator())
    bars_tf=CFG['main']
    if completed(f'gold_{bars_tf}'):
        gold=spark.read.parquet(str(GOLD/bars_tf))
        visible=gold.where(F.col('available_at')<=F.lit(str(end)).cast('timestamp')).orderBy(F.desc('available_at')).limit(240)
        rows=list(visible.toLocalIterator())[::-1]
        panels=[
            ('OHLC Mid',['mid_open','mid_high','mid_low','mid_close']),
            ('SMA et EMA',[f'sma_{n}' for n in CFG['sma']]+[f'ema_{n}' for n in CFG['ema']]),
            ('RSI',[x for x in FEATURES if x.startswith('rsi_')]),
            ('MACD',[x for x in FEATURES if x.startswith('macd_')]),
            ('Stochastique',[x for x in FEATURES if x.startswith('stoch_')]),
            ('CCI',[x for x in FEATURES if x.startswith('cci_')]),
            ('ATR',[x for x in FEATURES if x.startswith('atr_')]),
            ('Bollinger',[x for x in FEATURES if x.startswith(('bb_middle_','bb_upper_','bb_lower_'))]),
            ('Tick-OBV',[x for x in FEATURES if x.startswith('tick_obv')])]
        fig,axes=plt.subplots(3,3,figsize=(14,8),layout='constrained')
        for ax,(title,keys) in zip(axes.flat,panels):
            for key in keys:
                ax.plot([float(r[key]) if r[key] is not None else math.nan for r in rows],label=key)
            ax.set_title(title)
            ax.legend(fontsize=6)
        display(fig)
        plt.close(fig)
        count=gold.count()
        missing=gold.agg(*[F.count_if(F.col(x).isNull()).alias(x) for x in FEATURES]).first().asDict()
        fig,ax=plt.subplots(figsize=(11,4))
        ax.bar(range(len(missing)),[100*v/count for v in missing.values()])
        ax.set(ylabel='% indisponible',title=f'Valeurs absentes · {bars_tf}')
        ax.set_xticks(range(len(missing)),list(missing),rotation=90,fontsize=5)
        display(fig)
        plt.close(fig)
        keys=[x for x in CFG['features'] if x in gold.columns][:8]
        if keys:
            cor=visible.agg(*[F.corr(a,b).alias(f'{i}_{j}') for i,a in enumerate(keys)
                             for j,b in enumerate(keys)]).first().asDict()
            matrix=[[cor[f'{i}_{j}'] if cor[f'{i}_{j}'] is not None else math.nan
                     for j in range(len(keys))] for i in range(len(keys))]
            fig,ax=plt.subplots(figsize=(5,4))
            image=ax.imshow(matrix,vmin=-1,vmax=1,cmap='coolwarm')
            ax.set(xticks=range(len(keys)),xticklabels=keys,yticks=range(len(keys)),yticklabels=keys,title='Corrélations sur les 240 dernières barres')
            ax.tick_params(axis='x',rotation=30)
            fig.colorbar(image)
            display(fig)
            plt.close(fig)
    if vals:
        fig,axes=plt.subplots(1,3,figsize=(11,3))
        for ax,key in zip(axes,('ask_volume','bid_volume','spread')):
            ax.hist([float(r[key]) for r in vals if r[key] is not None],bins=35)
            ax.set_title(key)
        display(fig)
        plt.close(fig)
    minutes=list(spark.read.parquet(str(PROFILE/'minute_activity')).select('count').sample(.15,seed=7).limit(1000).toLocalIterator())
    if minutes:
        fig,ax=plt.subplots(figsize=(7,2.8))
        ax.hist([r['count'] for r in minutes],bins=25)
        ax.set_title('Distribution des ticks par minute sur l’échantillon')
        display(fig)
        plt.close(fig)
    if bench:
        fig,ax=plt.subplots(figsize=(8,2.8))
        ax.bar([f"{r['level']} / {r['shuffle_partitions']}" for r in bench],[r['ticks_per_second'] for r in bench])
        ax.set(ylabel='Ticks/s',title='Benchmark Spark')
        display(fig)
        plt.close(fig)

if '18' in CFG['steps']:
    show_figures()
print('Spark UI',spark.sparkContext.uiWebUrl if spark else 'consultation des rapports')
say('Les graphiques sont calculés à partir d’agrégats et de fenêtres bornées.')

## 19 · Catalogue des features

La formule, le préchauffage et les taux valides sont documentés par timeframe. Le statut numérique décrit une feature disponible à la clôture.

In [ ]:
def feature_desc(key):
    if key.startswith('sma_'):
        return 'Tendance','moyenne des clôtures',key.split('_')[1],'mid_close','USD/once',int(key.split('_')[1])
    if key.startswith('ema_'):
        n=int(key.split('_')[1])
        return 'Tendance','EMA alpha=2/(n+1); amorce SMA',str(n),'mid_close','USD/once',n
    if key.startswith('rsi_'):
        n=int(key.split('_')[1])
        return 'Momentum','100 − 100/(1 + gains RMA/pertes RMA)',str(n),'mid_close','0–100',n+1
    if key.startswith('macd'):
        fast,slow,signal=map(int,key.split('_')[-3:])
        formula='EMA rapide moins EMA lente'
        if 'signal' in key: formula='EMA du MACD'
        if 'hist' in key: formula='MACD moins signal'
        return 'Momentum',formula,f'{fast},{slow},{signal}','mid_close','USD/once',slow if key.startswith('macd_'+str(fast)) else slow+signal-1
    if key.startswith('stoch_'):
        n,d=map(int,key.split('_')[-2:])
        return 'Momentum','100 × (clôture − min bas)/(max haut − min bas)' if '_k_' in key else 'SMA de %K',f'{n},{d}','mid_high mid_low mid_close','0–100',n if '_k_' in key else n+d-1
    if key.startswith('cci_'):
        n=int(key.split('_')[1])
        return 'Momentum','Écart du prix typique / (0,015 × déviation absolue moyenne)',str(n),'mid_high mid_low mid_close','sans unité',n
    if key.startswith('atr_'):
        n=int(key.split('_')[1])
        return 'Volatilité','RMA du True Range',str(n),'mid_high mid_low mid_close','USD/once',n
    if key.startswith('bb_'):
        n,k=key.split('_')[-2:]
        formula={'middle':'SMA','upper':'SMA + k × écart-type de population','lower':'SMA − k × écart-type de population',
                 'bandwidth':'largeur / SMA','percent':'position relative dans la bande'}[key.split('_')[1]]
        return 'Volatilité',formula,f'{n},{k}','mid_close','sans unité' if 'width' in key or 'percent' in key else 'USD/once',int(n)
    if key.startswith('tick_obv'):
        lag=int(key.rsplit('_',1)[-1]) if '_delta_' in key else 0
        return ('Activité','Cumul signé de tick_count' if not lag else 'Tick-OBV courant moins Tick-OBV retardé',str(lag),
                'mid_close tick_count','ticks',lag+1)
    window,roc,adx=CFG['extra_window'],CFG['roc'],CFG['adx']
    formula=extra_doc[key]
    if key.startswith('distance_sma_'):
        p=int(key.rsplit('_',1)[1])
        return 'Prix',formula,str(p),'mid_close SMA','ratio',p
    if key.startswith('relative_atr_'):
        p=int(key.rsplit('_',1)[1])
        return 'Volatilité',formula,str(p),'mid_close ATR','ratio',p
    if key.startswith('wma_'):
        return 'Tendance',formula,str(window),'mid_close','USD/once',window
    if key in {'dmi_plus','dmi_minus','adx'}:
        return 'Tendance',formula,str(adx),'mid_high mid_low mid_close','0–100',2*adx-1 if key=='adx' else adx
    if key=='parabolic_sar':
        return 'Tendance',formula,f"{CFG['sar_step']},{CFG['sar_max']}",'mid_high mid_low mid_close','USD/once',2
    if key in {'roc','momentum','williams_r'}:
        p=window if key=='williams_r' else roc
        return 'Momentum',formula,str(p),'mid_high mid_low mid_close','USD/once' if key=='momentum' else '%',p if key=='williams_r' else p+1
    if key.startswith('rolling_') or key in {'z_score','skewness','kurtosis','autocorr_1'}:
        unit='USD²/once²' if key=='rolling_variance' else ('USD/once' if key.startswith('rolling_') and key!='rolling_drawdown' else 'ratio')
        return 'Statistiques',formula,str(window),'mid_close',unit,window
    if key.endswith('_variance') or key=='realized_volatility':
        return 'Volatilité',formula,'intrabar','OHLC Mid et transitions de ticks','ratio',1
    if key in {'simple_return','log_return','lagged_log_return','high_low_ratio','close_open_ratio','price_range','candle_body','upper_wick','lower_wick'}:
        warmup=3 if key=='lagged_log_return' else 2 if key.endswith('return') else 1
        unit='USD/once' if key in {'price_range','candle_body','upper_wick','lower_wick'} else 'ratio'
        return 'Prix',formula,'','OHLC Mid',unit,warmup
    if key in {'hour','weekday','month_of_year','asia','europe','us','overlapping_sessions','hour_sin','hour_cos'}:
        return 'Temps',formula,'UTC','bar_open_time','heure' if key=='hour' else 'indice',1
    units={'spread_abs':'USD/once','spread_rel':'ratio','spread_bps':'points de base','spread_volatility':'USD/once',
           'tick_frequency':'ticks/s','tick_imbalance':'ratio','volume_imbalance':'ratio','intertick_seconds':'s',
           'spread_change':'USD/once','quote_volume_change':'volume de cotation'}
    return 'Microstructure',formula,'','ticks agrégés Bid/Ask',units[key],2 if key.endswith('_change') else 1

extra_doc={
    'simple_return':'close/close précédent − 1','log_return':'log(close/close précédent)',
    'lagged_log_return':'rendement log de la barre précédente','momentum':'close − close retardé',
    'price_range':'high − low','candle_body':'close − open','upper_wick':'high − max(open,close)',
    'lower_wick':'min(open,close) − low','high_low_ratio':'high/low','close_open_ratio':'close/open',
    'adx':'RMA de DX issu de +DI et −DI','dmi_plus':'100 × RMA(+DM)/RMA(TR)','dmi_minus':'100 × RMA(−DM)/RMA(TR)',
    'parabolic_sar':'SAR précédent + accélération × (extrême − SAR), avec retournements',
    'roc':'100 × (close/close retardé − 1)','williams_r':'−100 × (haut maximum − close)/amplitude',
    'parkinson_variance':'log(high/low)^2/(4 log 2)','garman_klass_variance':'0,5 log(H/L)^2 − (2 log 2−1) log(C/O)^2',
    'rogers_satchell_variance':'log(H/O) log(H/C) + log(L/O) log(L/C)',
    'realized_variance':'Somme des carrés des log rendements intrabar ticks','realized_volatility':'Racine de variance réalisée',
    'rolling_variance':'Variance de population des clôtures','rolling_std':'Racine de la variance de population',
    'rolling_mean':'Moyenne glissante des clôtures','z_score':'Écart à la moyenne / écart-type',
    'skewness':'Troisième moment centré standardisé','kurtosis':'Quatrième moment standardisé moins 3',
    'autocorr_1':'Covariation retard 1 / somme centrée des carrés','rolling_q10':'Quantile linéaire 0,10',
    'rolling_q90':'Quantile linéaire 0,90','rolling_min':'Minimum glissant','rolling_max':'Maximum glissant',
    'rolling_drawdown':'close/max(close) glissant − 1','spread_abs':'Ask−Bid à la clôture','spread_rel':'Spread / Mid',
    'spread_bps':'10 000 × spread / Mid','spread_volatility':'Écart-type des spreads de ticks',
    'tick_frequency':'Nombre de ticks / durée nominale en secondes','tick_imbalance':'(upticks − downticks)/ticks',
    'volume_imbalance':'(somme vol Bid − somme vol Ask)/(somme vol Bid + somme vol Ask) sur paires disponibles',
    'spread_change':'Spread de clôture moins spread de clôture précédent',
    'quote_volume_change':'Variation de la somme des volumes de cotation Bid et Ask',
    'intertick_seconds':'Durée moyenne intrabar entre ticks en secondes','hour':'Heure UTC décimale','weekday':'Jour UTC lundi=0',
    'month_of_year':'Mois UTC 1–12','asia':'Heure UTC < 9','europe':'7 ≤ heure UTC < 16','us':'13 ≤ heure UTC < 22',
    'overlapping_sessions':'Plusieurs plages horaires actives','hour_sin':'sin(2π heure/24)','hour_cos':'cos(2π heure/24)'}
for n in CFG['sma']:
    extra_doc[f'distance_sma_{n}']=f'close/SMA {n} − 1'
for n in CFG['atr']:
    extra_doc[f'relative_atr_{n}']=f'ATR {n}/close'
extra_doc[f"wma_{CFG['extra_window']}"]='Moyenne des clôtures pondérée de 1 à n'

def feature_catalogue(measure_now):
    catalog=[]
    for tf in CFG['timeframes']:
        valid={}
        total=0
        if measure_now and completed(f'gold_{tf}'):
            frame=spark.read.parquet(str(GOLD/tf))
            measures=frame.agg(F.count('*').alias('_rows'),
                *[F.count(x).alias(x) for x in FEATURES]).first().asDict()
            total=measures.pop('_rows')
            valid=measures
        for key in FEATURES:
            category,formula,params,columns,unit,warmup=feature_desc(key)
            catalog.append(dict(feature=key,timeframe=tf,category=category,formula=formula,
                  parameters=params,source_columns=columns,dtype='float64',unit=unit,warmup=warmup,
                  nulls='NULL jusqu’à la disponibilité et aux conditions de qualité',available_at='Clôture de la bougie',
                  valid_pct=round(100*valid[key]/total,3) if total else None,
                  limits='Tick-OBV utilise le nombre de cotations' if key.startswith('tick_obv') else 'Barres incomplètes et gaps exclus par défaut'))
    return catalog

if active('19'):
    cata=feature_catalogue(True)
    put_json(reports/'feature_catalogue.json',cata)
    finish('catalogue_features',records=len(cata))
else:
    cata=read_json(reports/'feature_catalogue.json',[])
    status('catalogue_features')
print('Aperçu',[(r['feature'],r['timeframe'],r['valid_pct']) for r in cata[:5]])
say('Le catalogue complet est exporté en JSON dans les rapports.')

## 20 · Rapport final

Les effectifs sont ceux de la fenêtre et du mode exécutés. La mesure du Bronze d’origine et les mesures Silver sont distinctes.

In [ ]:
def snapshot_report():
    names=['legacy_identity','normalized','silver','validation','bars','gold','causality','dataset','tests','catalogue_features']
    stats={name:manifest(name) for name in names}
    return dict(
        mode=run_mode,period=[str(start),str(end)],elapsed_run_seconds=round(perf_counter()-started_at,3),
        bronze=dict(files=len(sources),historical_reference=sum(int(r['ticks_locaux']) for r in annual) if annual else None,
                    raw_histdata=sum(int(r['ticks_histdata_bruts'] or 0) for r in annual)),
        measured=stats, benchmark=read_json(bench_path,[]),
        runtime=dict(spark=spark.version if spark else None, cores=cores, workers=workers,
                     driver_gib=driver_gib, available_ram_start_gib=round(avail,2),
                     available_ram_end_gib=round(psutil.virtual_memory().available/2**30,2) if psutil else None,
                     spark_ui=spark.sparkContext.uiWebUrl if spark else None),
        declared_period_gaps=['2026-11-23 : référence JForex à couverture partielle'],
        outputs_bytes={name:size_of(path) for name,path in [('normalized',NORM),('silver',TICKS),
                       ('bars',BARS),('gold',GOLD),('datasets',BUILT)] if path.exists()},
        caveats=['Les résultats concernent la fenêtre exécutée. Les heures à couverture inconnue restent à vérifier.',
               'HistData hors périodes calibrées reste en Bronze. Les volumes de transactions sont indisponibles.',
               'Les conflits multi-sources demeurent identifiés dans Silver.'])
if active('20') and run_mode=='FULL' and CFG['benchmark_full']:
    if not completed('validation'):
        raise RuntimeError('Le benchmark historique nécessite une sortie Silver validée.')
    results=read_json(bench_path,[])
    if not any(row['level']=='full_silver_history' for row in results):
        history=benchmark([],start,end,int(spark_options['spark.sql.shuffle.partitions']),frame=read_ticks())
        assert history['ticks']==manifest('silver')['silver']
        put_json(bench_path,[*results,history | {'level':'full_silver_history'}])
report=snapshot_report()
if active('20'):
    put_json(reports/'rapport_final.json',report)
print('Bronze',report['bronze'])
for name,record in report['measured'].items():
    print(name, {k:v for k,v in record.items() if k not in {'run_key','files','schema','scaler_train'}} if record else 'non exécuté')
print('Octets par sortie',report['outputs_bytes'])
say('Les comptes Silver et les mesures de performance sont ceux des manifestes présents sur disque.')